# 第04章 矩阵的解剖：可填写Notebook

> 模板原件：第一次先复制到本章 `工作/` 目录，再改成本章实际组名，例如 `A1组-实验.ipynb`、`B2组-验证.ipynb`。原件保留；副本内同组链接也要同步改名。复用导入图时，将本章 `模板/图片/` 中的图复制到 `工作/图片/`，图片链接使用 `图片/文件名`。组号见本章 `README.md`，复制步骤见仓库根目录下 `教程/02-Notebook教材写作.md` 的“模板复制与命名”一节。

> A组任务：A1、A2各自独立完成本章四节的全部实验及配套正文、习题与解答；不预先分小节。完整初稿经B/C验证后，本章六组讨论各部分采用版本，再由D/E整合。

![三格漫画：矩阵体检发现敏感方向，右端微小扰动导致解明显变化，机器人质疑小残差。](../模板/图片/A-章节导入.svg)

*图04-1 · 给矩阵做一次体检。初始化概念示意；适用条件见下文，不作为实测结果。*

## 先看一幕：给矩阵做一次体检

求解器递来一份漂亮的报告：“残差几乎为零。”矩阵医生却拿起放大镜：如果输入轻轻动一下，解会不会跳很远？

**先猜，再验证：** 设 $A=\operatorname{diag}(1,10^{-10})$。仅把右端第二个分量增加 $10^{-10}$，解的第二个分量会改变多少？这与求解器报告的小残差是否矛盾？

**把故事翻译成数学：** 在这个对角例子中，解的第二分量增加1，二范数条件数为 $10^{10}$。图中体检对应对矩阵结构与扰动敏感性的分析；小残差和小解误差是不同主张，还要说明残差针对哪个右端计算。

> 状态：按64学时方案整理的可填写模板，正文、推导和各节完整实验待学生完成。已有漫画与起步代码是初始化材料，不代表学生成果。

**知识主题：** 线性方程组、特征值与SVD

**本章核心问题：** Ax=b如何高效求解？矩阵的本质结构是什么？

**只聚焦三个核心概念：** LU、CG、SVD

**先修知识：** 矩阵运算、内积、正定性与向量/矩阵范数

**课时：** 4节 × 2学时 = 8学时；全书八章共64学时。

### 本章四节安排

| 节 | 内容 | 学时 |
| --- | --- | --- |
| 4.1 | 直接法：LU与Cholesky | 2 |
| 4.2 | 迭代法：CG与预条件 | 2 |
| 4.3 | 特征值与SVD | 2 |
| 4.4 | AI关联：可微求解器、PCA、LoRA | 2 |

### 怎么填写本章

从第4.1节开始，把每处“【待填写】”替换为自己的讲解、推导和证据。正文负责连贯讲解；[模板-A组-实验.ipynb](A2组-实验.ipynb)按相同节号组织文字与代码，具体实现和实际输出以Notebook为准，正文引用相应小节并解释结果。两处共有的公式、参数和结论须同步。

每节保留“讲解—推导—实验—解释—讨论”，这些是填写栏，不是额外课时。小节内多个方法按提示控制深度；选读不扩大必修范围。

**已有起步实验的位置：** 4.3节。小残差与大解误差的对角矩阵示例用于联系奇异值和条件数；不代表已完成LU、CG或AI应用。

---


**Notebook填写方法：** 双击Markdown单元填写讲解；在Code单元写Python，Shift+Enter运行，Ctrl+S保存。带有“TODO/未完成”的单元只是位置提示，即使顺序执行没有报错，也不能算完成本节。


In [1]:
import sys
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)
print("Python:", sys.version.split()[0], "NumPy:", np.__version__)
print("这是可填写模板；有输出或无报错不代表各节已完成。")

Python: 3.12.10 NumPy: 1.26.4
这是可填写模板；有输出或无报错不代表各节已完成。


## 4.1 直接法：LU与Cholesky（2学时）

**本节目标：** 能按矩阵条件选择分解，并解释主元和三角求解。

### 概念讲解：把问题说明白

填写提示：带主元LU、对称正定矩阵的Cholesky、分解复用与多右端求解；避免显式求逆作为默认解法。

#### 4.1.1 从Gauss消元法到矩阵的LU分解

线性方程组 $Ax=b$ 的求解是科学计算中最基本的问题之一。最基本的思想便是模拟手动求解的通用法——Gauss消元法。Gauss消元法的基本思路是：通过对矩阵 $A$ 进行一系列的行变换处理，将矩阵变成一个上三角矩阵，即可从下向上逐次求解每一个未知数。然而，当求解规模大到手动计算难以完成任务时，我们需要使用机器辅助运算，因此要设计一种流程更加固定、计算结果更加稳定的算法，并尽可能地减小运行算法所需计算量的规模。

回顾Gauss消元法的操作过程：对 $A$ 进行的一系列操作可以整体地视为一个矩阵的线性变换 $T$，满足 $U=TA$ 为上三角矩阵，之后求 $Ux=Tb$ 的解即可。对于上三角矩阵 $U$，这一过程只需从最下面开始求解，然后逐次带入到上一个方程中即可得到方程的解。

注意到操作过程保证了 $T$ 其实是一个下三角矩阵，显然可逆。现在，假如我们进行一个简单的变形，就得到了 $A=T^{-1}U$。换句话说，可以看作将 $A$ 分解为一个下三角矩阵 $L=T^{-1}$ 和一个上三角矩阵 $U$ 的乘积：

$$
A=LU \tag{4.1}
$$

总结一下上述思路，我们接下来给出求解线性方程组的**直接法**的基本概念与核心思想：


求解线性方程组 $Ax=b$ 的直接法，是指直接通过有限步算术运算直接求方程的解的方法。直接法的核心思想是：把系数矩阵 $A$ 分解为若干结构简单的矩阵的乘积（如前面提到的下三角/上三角矩阵，或对角矩阵、正交矩阵等），从而化简为更加容易求解的形式。

最简单的分解是将 $A$ 写成 $A=LU$，其中 $L$ 为单位下三角矩阵（对角元全为 $1$），$U$ 为上三角矩阵。这种分解被称为LU分解。得到这个分解后，可以把 $Ax=b$ 的求解过程分为两步：

1. 解 $Ly=b$，得到 $y$；
2. 解 $Ux=y$，得到 $x$。

下面我们来估计两种运算方案的复杂度：Gauss消元法先做前向消元，第 $k$ 步约需进行 $(n-k)^2+(n-k)$ 次乘除法 $(k=1,2,\cdots,n-1)$，求和得 $\sum\limits_{k=1}^{n-1}[(n-k)^2+(n-k)]=\frac{n^3-n}{3}=O(n^3)$，而回代求解过程中的计算次数约为 $\frac{n(n+1)}{2}=O(n^2)$，故总复杂度为 $O(n^3)$；LU分解中第 $k$ 步约需 $(2n-2k+1)(k-1)$ 次乘除法，求和得 $\sum\limits_{k=1}^{n}(2n-2k+1)(k-1)=\frac{n(n-1)(2n-1)}{6}=O(n^3)$，分解后求解过程中还需要约 $n^2=O(n^2)$ 次运算，因此总复杂度也是 $O(n^3)$，但对于同一个矩阵 $A$ 来说，LU分解完成后，假如只改变 $b$ 再次求解，那么复杂度仅为 $O(n^2)$。因此，虽然LU分解在单次计算中的复杂度与Gauss消元法差别不大，但是当需要求解多个具有相同系数矩阵 $A$ 而 $b$ 不相同的方程组时，矩阵分解的结果即可反复使用，从而在多次运算中大大节省计算的复杂度。

事实上，**Gauss消元法与LU分解本质上是同一件事。** 消元过程通过行初等变换把 $A$ 化为上三角矩阵，这些初等变换的累积效果恰好构造出下三角矩阵 $L$。从矩阵角度看，每一步消元相当于左乘一个初等矩阵；所有初等矩阵的逆的乘积就给出了 $L$。具体地说，Gauss消元法第 $k$ 步相当于左乘一个单位下三角矩阵 $L_k$，这里

$$
L_k=
\begin{pmatrix}
1&&&&\\
&\ddots&&&\\
&&1&&\\
&&-m_{k+1,k}&1&\\
&&\vdots&&\ddots\\
&&-m_{n,k}&&&1
\end{pmatrix},
\qquad
m_{ik}=\frac{a_{ik}^{(k)}}{a_{kk}^{(k)}},
$$

最终得到

$$
L_{n-1}\cdots L_2L_1A=U,
$$

于是

$$
A=(L_{n-1}\cdots L_1)^{-1}U
=L_1^{-1}L_2^{-1}\cdots L_{n-1}^{-1}U.
$$

易知 $L_k^{-1}$ 就是 $L_k$ 的非对角元取相反数，因此也都是单位下三角矩阵。故乘积 $L=L_1^{-1}\cdots L_{n-1}^{-1}$ 也是单位下三角矩阵，且其第 $i$ 行第 $j$ 列（$i>j$）的元素恰为消元过程中使用的乘数 $m_{ij}$。这就说明了Gauss消元法与LU分解的等价关系。

**直接三角分解法**（Doolittle分解）是LU分解的另一种实现方式，它不需要中间的计算步骤，直接从 $A$ 的元素计算 $L$ 和 $U$ 的元素。设 $A=LU$，比较两边元素可得递推公式：

$$
\begin{cases}
u_{1i}=a_{1i}, & i=1,2,\cdots,n,\\[2pt]
l_{i1}=\frac{a_{i1}}{u_{11}}, & i=2,3,\cdots,n,
\end{cases}
$$

对于 $r=2,3,\cdots,n$：

$$
\begin{cases}
u_{ri}=a_{ri}-\sum\limits_{k=1}^{r-1}l_{rk}u_{ki}, & i=r,r+1,\dots,n,\\[4pt]
l_{ir}=\frac{1}{u_{rr}}\left(a_{ir}-\sum\limits_{k=1}^{r-1}l_{ik}u_{kr}\right), & i=r+1,\dots,n.
\end{cases}
$$

#### 4.1.2 LU分解的性质与改进

LU分解的存在性和唯一性依赖于 $A$ 的顺序主子式：若 $A$ 的所有顺序主子式均不为零，则存在唯一的单位下三角矩阵 $L$ 和非奇异上三角矩阵 $U$，使得 $A=LU$。下面我们用归纳法证明：

设 $A$ 的所有顺序主子式均不为零。当 $n=1$ 时结论显然。假设结论对 $n-1$ 阶矩阵成立，将 $A$ 分块为

$$
A=
\begin{pmatrix}
A_{11}&A_{12}\\
A_{21}&A_{22}
\end{pmatrix},
$$

其中 $A_{11}$ 为 $n-1$ 阶顺序主子矩阵。由归纳假设，$A_{11}=L_{11}U_{11}$。令

$$
L_{21}=A_{21}U_{11}^{-1},\qquad
U_{12}=L_{11}^{-1}A_{12},\qquad
U_{22}=A_{22}-L_{21}U_{12},
$$

则

$$
A=
\begin{pmatrix}
L_{11}&0\\
L_{21}&I
\end{pmatrix}
\begin{pmatrix}
U_{11}&U_{12}\\
0&U_{22}
\end{pmatrix}.
$$

由 $A$ 非奇异可知 $U_{22}$ 非奇异，这就得到了LU分解。

LU分解是具有唯一性的：若 $A=LU=\bar{L}\bar{U}$，则 $L^{-1}\bar{L}=U\bar{U}^{-1}$。左边是单位下三角矩阵，右边是上三角矩阵，因此两边都必须是单位矩阵，故 $L=\bar{L}$，$U=\bar{U}$。

理论上，只要顺序主子式都不为零，LU 分解就存在且唯一。而当某个顺序主子式为零时，消元过程就会出现零主元（事实上，Gauss消元法也会遇到类似的问题，因为Gauss消元法能顺利完成的充要条件就是 $A$ 的所有顺序主子式都不为零。）；而当主元虽然非零但绝对值极小时，用它作除数会导致舍入误差被急剧放大。为了说明这一点，考虑一个简单但极端的例子：取 $A=\begin{pmatrix}0&1\\1&0\end{pmatrix}$。尽管 $A$ 非奇异，但 $a_{11}=0$，第一步消元就无法进行。

更隐蔽的情况是主元接近零：消元过程可以继续，但计算结果将可能严重失真。例如，取三位有效数字计算

$$
\begin{pmatrix}
0.001&2.000\\
1.000&3.000
\end{pmatrix}
\begin{pmatrix}
x_1\\
x_2
\end{pmatrix}
=
\begin{pmatrix}
2.001\\
4.000
\end{pmatrix}.
$$

精确解为 $x_1=1$，$x_2=1$。但若用 $0.001$ 作主元消元，乘数为 $1000$，第二个方程变为 $3.000-2000=-1997$，舍入到三位有效数字后精度损失极大，计算出的 $x_1$ 可能完全错误。

为了避免放大舍入误差，解决方法是预先做行的交换，把绝对值较大的元素换到主元位置。这就是列主元消去法（也称为部分选主元）。

列主元LU分解的矩阵形式为

$$
PA=LU,
$$

其中 $P$ 是置换矩阵，记录行交换的信息。$L$ 仍然是单位下三角矩阵，但它的元素在交换行之后会相应改变，且满足 $|l_{ij}|\leqslant 1$。这一性质保证了消元过程中乘数的绝对值不超过 $1$，从而避免舍入误差的过度放大。对于一般非奇异矩阵，列主元LU分解总是可行的。

在实际编程中，$L$ 和 $U$ 的元素可以直接存放在 $A$ 的相应位置，不需要额外存储空间。这种分解法大约需要 $\frac{n^3}{3}$ 次乘除法，与Gauss消元法计算量相同。

#### 4.1.3 对称正定矩阵的Cholesky分解

下面我们考虑实际问题中经常遇到的一类重要矩阵——**对称正定矩阵**，在最小二乘问题、偏微分方程的数值求解、函数的Hessian矩阵等问题中都经常能见到这类矩阵的影子。对称正定矩阵分解比较特殊：若 $A$ 对称且正定，则存在唯一的对角元全为正的下三角矩阵 $L$，使得 $A=LL^T$。这就是Cholesky分解（也称为平方根法）。Cholesky分解不需要选主元，因为对称正定矩阵的所有顺序主子式都为正，消元过程中不会出现零或小主元。与LU分解相比，Cholesky分解的计算量大约减半（约 $\frac{n^3}{6}$ 次乘除法），存储量也减半（只需存储 $L$ 的下三角部分）。它的代价是需要 $n$ 次开平方运算。

下面我们推导Cholesky分解的递推公式：设 $A=LL^T$，比较两边元素。对于 $j\leqslant i$，有

$$
a_{ij}=\sum_{k=1}^{j}l_{ik}l_{jk}
=\sum_{k=1}^{j-1}l_{ik}l_{jk}+l_{ij}l_{jj}.
$$

当 $i=j$ 时，

$$
a_{jj}=\sum_{k=1}^{j-1}l_{jk}^2+l_{jj}^2
\quad\Longrightarrow\quad
l_{jj}=\sqrt{a_{jj}-\sum_{k=1}^{j-1}l_{jk}^2}.
$$

当 $i>j$ 时，

$$
l_{ij}=\frac{1}{l_{jj}}
\left(
a_{ij}-\sum_{k=1}^{j-1}l_{ik}l_{jk}
\right).
$$

由于 $A$ 正定，根号下的量恒为正，因此 $l_{jj}>0$ 有定义。

而在数值稳定性方面，也很容易推出Cholesky分解是稳定的：由 $a_{jj}=\sum\limits_{k=1}^j l_{jk}^2$ 可得 $l_{jk}^2\leqslant a_{jj}$，因此 $\max\limits_{j,k}\{l_{jk}^2\}\leqslant \max\limits_{1\leqslant j\leqslant n}\{a_{jj}\}$。这说明分解过程中 $l_{jk}$ 不会无界增长，且对角元素 $l_{jj}$ 恒为正数。因此，不选主元素的Cholesky分解是一个数值稳定的方法。

进一步地，为了避免开方运算，可以使用**改进的平方根法**（$LDL^T$ 分解）：

$$
A=LDL^T,
$$

其中 $L$ 为单位下三角矩阵，$D$ 为对角矩阵。按行计算 $L$ 和 $D$ 的公式为：

$$
d_1=a_{11},
$$

对于 $i=2,3,\dots,n$：

$$
\begin{cases}
t_{ij}=a_{ij}-\sum\limits_{k=1}^{j-1}t_{ik}l_{jk}, & j=1,2,\dots,i-1,\\[4pt]
l_{ij}=\frac{t_{ij}}{d_j}, & j=1,2,\dots,i-1,\\[4pt]
d_i=a_{ii}-\sum\limits_{k=1}^{i-1}t_{ik}l_{ik}.
\end{cases}
$$

改进的平方根法与Cholesky分解计算量差不多，但不再需要开方计算。

#### 4.1.4 三对角矩阵的追赶法

以上介绍的分解方法都针对一般的稠密矩阵，如果矩阵还有更特殊的结构，比如说非零元集中在对角线附近的矩阵，可以进一步采用简化的分解方案。**三对角矩阵**这种只有三条对角线上的元素非零的矩阵正是这样一种特殊而又重要的矩阵，它经常出现在常微分方程边值问题、三次样条插值和热传导方程的差分格式中。三对角矩阵的LU分解只需要 $O(n)$ 次运算，而不是 $O(n^3)$。这一方法称为追赶法（Thomas算法）。追赶法的矩阵分解形式也是 $A=LU$，但这里 $L$ 是下双对角矩阵（次对角线为 $l_i$，对角元为 $1$），$U$ 是上双对角矩阵（对角线为 $u_i$，上次对角线为 $c_i$）。分解的递推公式只需要将 $A=LU$ 按元素展开：首先有 $b_1=u_1$，且 $c_1=u_1\beta_1$，$a_i=l_i u_{i-1}$，$b_i=l_i c_{i-1}+u_i$，由此解出 $l_i=\frac{a_i}{u_{i-1}}$，$u_i=b_i-l_ic_{i-1}\ (i=2,\dots,n)$。

求解过程分为“追”和“赶”两步：前代求解 $Ly=d$ 得到 $y_i$，回代求解 $Ux=y$ 得到 $x_i$。追赶法能够顺利进行的条件是三对角矩阵满足对角占优条件：$|b_1|>|c_1|$，$|b_i|\geqslant|a_i|+|c_i|\ (i=2,\dots,n-1)$，$|b_n|>|a_n|$。在这一条件下，所有 $u_i$ 非零且 $|l_i|<1$，计算过程稳定。追赶法的运算量仅为 $5n-4$ 次乘除法，加减运算约 $3n$ 次。

在对角占优条件下，可以归纳证明 $|l_i|<1$，从而保证计算过程的稳定性：设 $|b_1|>|c_1|>0$，$|b_i|\geqslant |a_i|+|c_i|$，$|b_n|>|a_n|>0$。令 $\beta_1=\frac{c_1}{b_1}$，则 $|\beta_1|<1$。假设 $|\beta_{i-1}|<1$，则

$$
|u_i|=|b_i-a_i\beta_{i-1}|
\geqslant |b_i|-|a_i|\cdot|\beta_{i-1}|
>|b_i|-|a_i|
\geqslant |c_i|>0,
$$

且 $|\beta_i|=\left|\frac{c_i}{u_i}\right|<1$。由归纳法知所有 $|\beta_i|<1$，因此回代过程中误差不会放大，追赶法是稳定的。


#### 4.1.5 误差分析

直接法在不考虑舍入误差时给出的是精确解，且计算量基本固定。它的优势在于稳定、可靠，适用于中小规模稠密矩阵，也适用于某些具有特殊结构的大型稀疏矩阵。不过它的局限在于当矩阵规模很大，分解过程会产生大量“填充”（原本为零的位置在分解后变成非零），存储和计算代价都迅速增长。

当然，上面的讨论都建立在精确算术的前提下。实际计算中，浮点舍入误差不可避免，因此需要回答一个更基本的问题：**输入端的微小扰动，会在解中被放大多少倍？**

我们通过计算来分析：设 $A$ 非奇异，考虑方程 $Ax=b$。若假定 $A$ 精确，右端有微小扰动 $\delta b$，则实际求解的是扰动方程组 $A\widetilde{x}=b+\delta b.$ 记真解为 $x=A^{-1}b$，解的扰动为 $\delta x=\widetilde{x}-x$. 带入得 $
\delta x=A^{-1}\delta b.$ 两边取范数，并利用 $\|b\|\leqslant \|A\|\,\|x\|$，可得$\frac{\|\delta x\|}{\|x\|}\leqslant \|A^{-1}\|\,\|A\|\,\frac{\|\delta b\|}{\|b\|}.$

这里右端出现的因子
$$
\kappa(A)\triangleq \|A^{-1}\|\,\|A\|
$$

称为矩阵 $A$ 的**条件数**。它刻画了右端相对扰动到解相对扰动的最大放大倍数。可以看出条件数越大，解对扰动越敏感，问题越病态。常用的条件数即 $\kappa_2(A)=\|A^{-1}\|_2\|A\|_2$，称为谱条件数。后续若无特殊说明，条件数均指的是 $\kappa_2(A)$. 关于条件数的更多相关知识留到后续章节介绍。

回到本节介绍的直接法，误差分析给出以下结论：

1. **无主元 LU 可能放大误差。** 小主元导致乘子很大，舍入误差被急剧放大；列主元 LU 使 $|l_{ij}|\leqslant 1$，避免乘子无界增长。
2. **Cholesky 分解稳定。** 由 $a_{jj}=\sum\limits_{k=1}^{j}l_{jk}^2$ 可得 $l_{jk}^2\leqslant a_{jj}$，元素不会无界增长，无需选主元。
3. **追赶法依赖对角占优。** 对角占优时 $|\beta_i|<1$，误差不放大；非对角占优时可能放大。
4. **选主元只改善算法稳定性，不改善问题病态性。** 条件数是问题本身的属性，与算法无关。对病态矩阵，即使残差 $r=b-A\hat{x}$ 很小，解误差仍可能很大，因为

$$
\frac{\|\hat{x}-x\|}{\|x\|}\leqslant \kappa(A)\frac{\|r\|}{\|b\|}.
$$

**易错点提示：** 并非任意矩阵都可以进行无主元的LU分解；对非正定矩阵不能无条件使用Cholesky分解。列主元消去法虽然能解决小主元问题，但它不能改善矩阵本身的病态性。对于病态矩阵，选主元只能避免因算法引起的额外误差，而无法消除问题固有的敏感性。此外，追赶法的对角占优条件只是充分条件，不是必要条件；某些非对角占优的三对角矩阵也可能稳定求解，但需要额外验证。

### 关键推导：让结论有依据

推导任务：手算一个小矩阵的消元与回代，明确置换矩阵约定；写明Cholesky分解适用条件。

**推导 1：手算一个小矩阵的列主元 LU 分解。** 取

$$
A=\begin{bmatrix}0&2&1\\1&0&3\\2&1&1\end{bmatrix},\quad
x_\star=\begin{bmatrix}1\\-2\\3\end{bmatrix},\quad
b=\begin{bmatrix}-1\\10\\3\end{bmatrix}.
$$

约定置换矩阵 $P$ 满足 $PA=LU$，即 $P$ 左乘 $A$ 后得到行交换后的矩阵。

**第一步：选主元并消元。** 原位置第一个主元 $a_{11}=0$，不能直接相除。在第一列中选取绝对值最大的元素，即 $a_{31}=2$，交换第 $1$、$3$ 行，得到

$$
\begin{bmatrix}2&1&1\\1&0&3\\0&2&1\end{bmatrix}.
$$

用乘子 $m_{21}=\frac{1}{2}$ 消去第二行首项：

$$
\begin{bmatrix}2&1&1\\0&-\frac12&\frac52\\0&2&1\end{bmatrix}.
$$

此时第二列主元为 $-\frac12$，第三行第二列元素为 $2$。由于 $|2|>|-\frac12|$，需要再次换行。

**第二步：第二次换行并消元。** 交换第 $2$、$3$ 行，得到

$$
\begin{bmatrix}2&1&1\\0&2&1\\0&-\frac12&\frac52\end{bmatrix}.
$$

用乘子 $m_{32}=-\frac14$ 消去第三行第二列元素：

$$
\begin{bmatrix}2&1&1\\0&2&1\\0&0&\frac{11}{4}\end{bmatrix}.
$$

**第三步：写出 $P$、$L$、$U$。** 两次行交换对应的置换矩阵为

$$
P=\begin{bmatrix}0&0&1\\1&0&0\\0&1&0\end{bmatrix},
$$

它满足 $P\ne P^T$。上三角矩阵为

$$
U=\begin{bmatrix}2&1&1\\0&2&1\\0&0&\frac{11}{4}\end{bmatrix}.
$$

下三角矩阵 $L$ 的单位对角元为 $1$，非对角元为消元过程中使用的乘子。注意：第二次换行发生在消元之后，$L$ 中已经存好的乘子也要相应交换。具体地，第一次消元得到 $m_{21}=\frac12$，第二次消元得到 $m_{32}=-\frac14$；交换第 $2$、$3$ 行后，$L$ 中第 $2$、$3$ 行的乘子也交换位置，得到

$$
L=\begin{bmatrix}1&0&0\\0&1&0\\\frac12&-\frac14&1\end{bmatrix}.
$$

**第四步：核验 $PA=LU$。** 直接计算

$$
PA=\begin{bmatrix}2&1&1\\0&2&1\\1&0&3\end{bmatrix},
$$

而

$$
LU=\begin{bmatrix}1&0&0\\0&1&0\\\frac12&-\frac14&1\end{bmatrix}
\begin{bmatrix}2&1&1\\0&2&1\\0&0&\frac{11}{4}\end{bmatrix}
=\begin{bmatrix}2&1&1\\0&2&1\\1&0&3\end{bmatrix}.
$$

两边相等，验证 $PA=LU$ 成立。这里特意选择两次换行，是因为若只交换一次，$P=P^T$，错误的置换约定可能被样例掩盖。

**推导 2：回代、多右端求解与误差核验。** 由 $PA=LU$，原方程 $Ax=b$ 等价于

$$
LUx=Pb.
$$

令 $y=Ux$，则求解分为两步：

1. 解 $Ly=Pb$，得到 $y$；
2. 解 $Ux=y$，得到 $x$。

下三角前代公式为

$$
y_i=\frac{1}{L_{ii}}\left((Pb)_i-\sum_{j=1}^{i-1}L_{ij}y_j\right),\qquad i=1,2,\dots,n.
$$

上三角回代公式为

$$
x_i=\frac{1}{U_{ii}}\left(y_i-\sum_{j=i+1}^{n}U_{ij}x_j\right),\qquad i=n,n-1,\dots,1.
$$

由于 $L$ 是单位下三角矩阵，$L_{ii}=1$，前代中不涉及除法；回代中 $U_{ii}$ 一般不为零，否则 $A$ 奇异。

多右端情形可以整体处理。若右端为矩阵 $B$，则求解 $AX=B$ 只需把上述两步分别作用于各列：

$$
LY=PB,\qquad UX=Y.
$$

这样分解一次后，每个右端只增加 $O(n^2)$ 的求解成本，而不需要重新做 $O(n^3)$ 的分解。

误差核验通常计算残差与解误差。残差为

$$
r=b-Ax.
$$

若已知真解 $x_\star$，还可计算解误差 $\|x-x_\star\|_2$。在列主元 LU 分解下，残差通常可达到机器精度量级；但残差小不一定解误差小，因为解误差还受条件数影响：

$$
\frac{\|x-x_\star\|_2}{\|x_\star\|_2}
\leqslant \kappa_2(A)\frac{\|r\|_2}{\|b\|_2}.
$$

这一关系说明，当 $A$ 病态时，即使残差很小，解误差仍可能较大。

**推导 3：对称正定矩阵与 Cholesky 分解。** 实矩阵 $H$ 对称正定是指 $H=H^T$，并且对任意非零向量 $z$ 都有 $z^THz>0$。这时存在对角元为正的下三角矩阵 $L$，使

$$
H=LL^T.
$$

下面通过比较矩阵元素推导 $L$ 的逐列计算公式。设 $L$ 的下三角元素为 $l_{ij}$（$i\geqslant j$），比较 $H=LL^T$ 两边元素。对 $i\geqslant j$，有

$$
H_{ij}=\sum_{k=1}^{j}l_{ik}l_{jk}
=\sum_{k=1}^{j-1}l_{ik}l_{jk}+l_{ij}l_{jj}.
$$

当 $i=j$ 时，上式变为

$$
H_{jj}=\sum_{k=1}^{j-1}l_{jk}^2+l_{jj}^2,
$$

因此对角元为

$$
l_{jj}=\sqrt{H_{jj}-\sum_{k=1}^{j-1}l_{jk}^2}.
$$

当 $i>j$ 时，由同一比较式解出非对角元：

$$
l_{ij}=\frac{1}{l_{jj}}
\left(
H_{ij}-\sum_{k=1}^{j-1}l_{ik}l_{jk}
\right).
$$

正定性保证精确算术中根号内的量恒为正，因此 $l_{jj}>0$ 有定义，逐列计算可以顺利进行。

例如

$$
H=\begin{bmatrix}4&2&2\\2&5&1\\2&1&3\end{bmatrix}
=\begin{bmatrix}2&0&0\\1&2&0\\1&0&\sqrt2\end{bmatrix}
\begin{bmatrix}2&1&1\\0&2&0\\0&0&\sqrt2\end{bmatrix}.
$$

因右侧下三角因子可逆，对非零 $z$ 有 $z^THz=\|L^Tz\|_2^2>0$，这也给出了本例正定性的直接证明。反例 $\begin{bmatrix}1&2\\2&1\end{bmatrix}$ 的特征值是 $3$ 和 $-1$，虽对称却不正定，第二个开方量为 $-3$。

标准无主元 Cholesky 分解不能无条件用于仅半正定、非正定或不对称矩阵；接近奇异时，还可能因舍入失去数值正定性。

### 实验设计：先预测，再运行

实验任务：用小型一般矩阵及正定矩阵求解，与可信库函数比较；加入需交换主元的例子；验证 Cholesky 分解的稳定性和追赶法的对角占优条件。

四组子实验：

- **E4.1-1**：$3\times 3$ 一般矩阵的列主元 LU 分解，检查两次换行、$PA=LU$、前代回代、解误差与残差。
- **E4.1-2**：$4\times 4$ 对称正定矩阵的 Cholesky 分解，检查 $\frac{\lVert A-LL^T\rVert_F}{\lVert A\rVert_F}$、多右端相对解误差、元素不增长。
- **E4.1-3**：小主元矩阵 $A_3=\begin{pmatrix}0.001&2\\1&3\end{pmatrix}$，比较无主元 LU 与列主元 LU 的解误差。
- **E4.1-4**：三对角矩阵的追赶法，分别取对角占优与非对角占优各一例，比较残差与解误差。

| 要填写的项目 | 本组内容 |
| --- | --- |
| 要回答的问题与运行前预测 | **问题：** 1）无主元 LU 何时失败或严重失真？2）列主元 LU 是否总能给出可信解？3）Cholesky 对非对称或非正定矩阵的行为？4）追赶法在对角占优条件下的稳定性如何体现？**预测：** 对一般矩阵，无主元 LU 在遇到零或小主元时会失败或精度严重下降；列主元 LU 能成功；Cholesky 仅对对称正定矩阵适用，对非正定矩阵会因根号下为负而失败。三对角矩阵的追赶法在非对角占优时可能不稳定；对角占优时，$\lvert\beta_i\rvert<1$，回代误差不放大。直接三角分解法与列主元 LU 分解得到相同解，但计算量相同；改进的平方根法与 Cholesky 得到相同解，但不需要开方。 |
| 输入、参数、单位、数据来源与划分 | 输入：随机生成的 $4\times 4$ 一般矩阵 $A_1$；对称正定矩阵 $A_2=MM^T$（$M$ 随机下三角且对角元为正）；需要交换主元的矩阵 $A_3=\begin{pmatrix}0.001&2\\1&3\end{pmatrix}$；三对角矩阵 $A_4$（对角占优和非对角占优各一例）；Hilbert 矩阵 $H_4$（用于观察病态性）。数据来源：本组用固定种子生成。 |
| 独立参考解/基线及选择理由 | 参考解：用 `numpy.linalg.solve` 或 MATLAB 的 `\` 作为基线。选择理由：这些库函数内部使用列主元 LU，经过了充分测试，可作为“可信解”。 |
| 误差指标、容差及其依据 | 指标：分解误差 $\lVert A-LU\rVert_F$ 或 $\lVert A-LL^T\rVert_F$; 残差 $\frac{\lVert A-LL^T\rVert_F}{\lVert A\rVert_F}$; 解误差 $\lVert\widetilde{x}-x_{\text{true}}\rVert_2$（当已知真解时）；条件数 $\kappa_2(A)$。容差：约 $10^{-12}$ 量级（双精度）。 |
| 环境、种子、设备与运行预算 | 环境：Python 3 + NumPy，或 MATLAB。种子：本组固定。设备：个人电脑即可。预算：秒级。 |


```python
import numpy as np
from scipy.linalg import lu, cholesky, solve_triangular

np.set_printoptions(precision=6, suppress=True)

# E4.1-1: 需要两次换行的 3x3 例子
A = np.array([[0., 2., 1.],
              [1., 0., 3.],
              [2., 1., 1.]])
x_true = np.array([1., -2., 3.])
b = A @ x_true

P_scipy, L, U = lu(A)
P = P_scipy.T

print("PA - LU 的 Frobenius 误差:", np.linalg.norm(P @ A - L @ U, 'fro'))
print("P =\n", P)
print("L =\n", L)
print("U =\n", U)

y = solve_triangular(L, P @ b, lower=True, unit_diagonal=True)
x = solve_triangular(U, y, lower=False)
print("前代 y =", y)
print("回代 x =", x)
print("解误差 ||x - x_true||_2 =", np.linalg.norm(x - x_true, 2))
print("相对残差 ||b - A x||_2 / ||b||_2 =",
      np.linalg.norm(b - A @ x, 2) / np.linalg.norm(b, 2))

# E4.1-2: 对称正定矩阵与 Cholesky
rng = np.random.default_rng(2026)
n = 4
M = np.tril(rng.normal(size=(n, n)))
M[np.diag_indices(n)] = np.abs(M[np.diag_indices(n)]) + 0.5
A_spd = M @ M.T

L_chol = cholesky(A_spd, lower=True)
print("Cholesky 分解误差 ||A - L L^T||_F / ||A||_F =",
      np.linalg.norm(A_spd - L_chol @ L_chol.T, 'fro') / np.linalg.norm(A_spd, 'fro'))
print("max |L_ij| =", np.max(np.abs(L_chol)))
print("max A_jj    =", np.max(np.diag(A_spd)))

X_true = rng.normal(size=(n, 3))
B = A_spd @ X_true
Y = solve_triangular(L_chol, B, lower=True)
X_hat = solve_triangular(L_chol.T, Y, lower=False)
print("多右端相对解误差 ||X_hat - X_true||_F / ||X_true||_F =",
      np.linalg.norm(X_hat - X_true, 'fro') / np.linalg.norm(X_true, 'fro'))

# E4.1-3: 小主元与列主元对比
A_bad = np.array([[0.001, 2.000],
                  [1.000, 3.000]])
b_bad = np.array([2.001, 4.000])
x_bad_true = np.array([1.0, 1.0])

def lu_no_pivot(A):
    """返回单位下三角 L 和上三角 U，满足 A = L U。遇到零主元时抛错。"""
    A = A.astype(float).copy()
    n = A.shape[0]
    L = np.eye(n)
    U = A.copy()
    for k in range(n - 1):
        if abs(U[k, k]) < 1e-15:
            raise ValueError("零主元，无主元 LU 失败")
        for i in range(k + 1, n):
            m = U[i, k] / U[k, k]
            L[i, k] = m
            U[i, k:] -= m * U[k, k:]
    return L, U

try:
    L_bad, U_bad = lu_no_pivot(A_bad)
    y_bad = solve_triangular(L_bad, b_bad, lower=True, unit_diagonal=True)
    x_bad = solve_triangular(U_bad, y_bad, lower=False)
    print("无主元 LU 解:", x_bad)
    print("无主元 LU 解误差:", np.linalg.norm(x_bad - x_bad_true, 2))
except ValueError as e:
    print("无主元 LU 失败:", e)

P_bad, L_bad_p, U_bad_p = lu(A_bad)
x_bad_p = np.linalg.solve(A_bad, b_bad)
print("列主元/库函数解:", x_bad_p)
print("列主元解误差:", np.linalg.norm(x_bad_p - x_bad_true, 2))

# E4.1-4: 追赶法
def thomas(a, b, c, f):
    """三对角矩阵追赶法。a 次对角线长度 n-1，b 主对角线长度 n，
    c 上对角线长度 n-1，f 右端长度 n。返回解 x。"""
    n = len(b)
    if n == 1:
        return np.array([f[0] / b[0]])
    beta = np.zeros(n - 1)
    y = np.zeros(n)
    x = np.zeros(n)
    beta[0] = c[0] / b[0]
    y[0] = f[0] / b[0]
    for i in range(1, n - 1):
        denom = b[i] - a[i - 1] * beta[i - 1]
        beta[i] = c[i] / denom
        y[i] = (f[i] - a[i - 1] * y[i - 1]) / denom
    denom = b[n - 1] - a[n - 2] * beta[n - 2]
    y[n - 1] = (f[n - 1] - a[n - 2] * y[n - 2]) / denom
    x[n - 1] = y[n - 1]
    for i in range(n - 2, -1, -1):
        x[i] = y[i] - beta[i] * x[i + 1]
    return x

n_tri = 6
a_tri = -np.ones(n_tri - 1)
b_tri = 3 * np.ones(n_tri)
c_tri = -np.ones(n_tri - 1)
x_tri_true = np.arange(1, n_tri + 1, dtype=float)
A_tri = np.diag(b_tri) + np.diag(a_tri, -1) + np.diag(c_tri, 1)
f_tri = A_tri @ x_tri_true
x_tri_hat = thomas(a_tri, b_tri, c_tri, f_tri)
print("追赶法解误差:", np.linalg.norm(x_tri_hat - x_tri_true, 2))
print("追赶法相对残差:",
      np.linalg.norm(f_tri - A_tri @ x_tri_hat, 2) / np.linalg.norm(f_tri, 2))
```



In [2]:
# 4.1 输入与独立参考
# TODO：填写本节输入、参数、种子和独立参考；不得用待验证算法自身输出冒充真值。
# 在这里补充代码；完成后删除这条占位提示。
print('4.1 输入与独立参考：未完成，请填写并实际运行。')


4.1 输入与独立参考：未完成，请填写并实际运行。


In [3]:
# 4.1 方法实现与运行
# TODO：用小型一般矩阵及正定矩阵求解，与可信库函数比较；加入需交换主元的例子。
# 在这里补充代码；完成后删除这条占位提示。
print('4.1 方法实现与运行：未完成，请填写并实际运行。')


4.1 方法实现与运行：未完成，请填写并实际运行。


In [4]:
# 4.1 误差图表与比较
# TODO：报告分解关系的误差、残差、已知真解误差及计算量说明。
# 在这里补充代码；完成后删除这条占位提示。
print('4.1 误差图表与比较：未完成，请填写并实际运行。')


4.1 误差图表与比较：未完成，请填写并实际运行。


### 结果解释与本节结论

应提供的证据：报告分解关系的误差、残差、已知真解误差及计算量说明；比较无主元LU与列主元LU在病态例子上的差异；验证Cholesky分解的稳定性（元素不增长）；验证追赶法在对角占优条件下的稳定性。

- **E4.1-1**：$\frac{\lVert PA-LU\rVert_F}{\lVert A\rVert_F}$ 约为 $10^{-16}$ 量级；解误差约 $10^{-16}$；相对残差约 $10^{-16}$；条件数 $\kappa_2(A)$ 记录在案。若数值大于阈值，应检查置换矩阵约定与库函数返回值，而不是立即断定算法错误。
- **E4.1-2**：$\frac{\lVert A-LL^T\rVert_F}{\lVert A\rVert_F}$ 约 $10^{-17}$ 量级；多右端相对解误差约 $10^{-16}$；$\max\lvert L_{ij}\rvert$ 不超过 $\sqrt{\max_j A_{jj}}$，验证“元素不增长”。若分解误差明显超出阈值，需核对 `cholesky` 的 `lower` 参数与转置约定。
- **E4.1-3**：无主元LU的解误差远大于列主元LU（具体数值取决于浮点环境），说明小主元确实放大舍入误差；列主元LU与 `numpy.linalg.solve` 一致。若两者恰好都接近真解，应报告“本例未能触发误差放大”，而不是断言小主元无害。
- **E4.1-4**：对角占优例的追赶法相对残差约 $10^{-16}$；非对角占优例的残差可能明显偏大，与预测一致。需要强调的是，非对角占优只是充分条件不成立，不等于算法一定失败。

**本节结论：** 直接法的核心是矩阵分解。LU分解适用于一般非奇异矩阵，但需要选主元以保证数值稳定性；Cholesky分解适用于对称正定矩阵，乘除法次数约 $n^3/6$，约为LU的一半，且无需选主元；追赶法适用于三对角矩阵，乘除法次数约 $5n-4$，计算量降至 $O(n)$。选择哪种分解，取决于矩阵的结构和条件数。对于病态矩阵，选主元只能避免算法引起的不稳定，不能改善问题本身的敏感性。

### 易错点与讨论

待核验的风险：任意矩阵都可无主元LU，或对非正定矩阵无条件使用Cholesky分解。


- **反例 1（无主元LU失效）**：$A=\begin{pmatrix}0&1\\1&0\end{pmatrix}$，$a_{11}=0$，无主元LU第一步就除以零。用E4.1-3的小主元矩阵可进一步看到，即使主元非零，只要它非常小，无主元LU的解误差也会显著放大。
- **反例 2（Cholesky不适用）**：$A=\begin{pmatrix}1&2\\2&1\end{pmatrix}$ 对称但特征值为 $3$ 和 $-1$，第二个开方量 $a_{22}-l_{21}^2=1-4=-3<0$，Cholesky失败。
- **结论**：Cholesky需要对称正定；无主元LU需要所有顺序主子式不为零。选主元只解决算法稳定性，不改善病态性。

**随堂提问：** 设计需要主元交换的手算题，再讨论是否适用Cholesky分解。给定一个非对角占优但实际可解的三对角矩阵，讨论追赶法是否仍然可用，并给出一种判断方法。



- **明确条件**：题目给出一个 $3\times 3$ 矩阵 $A_0$，第一列首元为零，但 $A_0$ 非奇异；同时给出一个对称但非正定的矩阵 $B_0$；再给出一个非对角占优的三对角矩阵 $T_0$。
- **同学可能的回答**：一部分同学可能直接说“可以LU，所以能Cholesky”；另一部分可能认为“非对角占优就一定不能用追赶法”。
- **本组最终解释**：第一题需要列主元才能完成LU；第二题Cholesky失败，因为 $B_0$ 非正定；第三题中，非对角占优只是追赶法稳定性的充分条件不成立，不等于算法一定失败。可以逐步检查 $\lvert\beta_i\rvert<1$ 或 $\lvert u_i\rvert>0$，或者直接做分解并观察是否出现零主元。若时间允许，可以在Notebook中对 $T_0$ 做一次追赶法分解，打印 $\beta_i$ 与 $u_i$，用实际数值支持判断。

### 4.1 实际运行记录

【待填写】运行日期、完整环境/新增依赖版本、代码状态、输出位置、误差与结论。对照上面的证据要求逐项解释；未运行写未运行。AI辅助内容及人工改动记录在本章模板-A组-AI对话记录.md。

## 4.2 迭代法：CG与预条件（2学时）

**本节目标：** 能说明CG的适用条件和预条件改善的对象。

### 概念讲解：把问题说明白

4.1节所介绍的直接法运算量为 $O(n^3)$，在矩阵规模很大时，将会面临存储和计算量急剧增长，这时直接法通常不是最佳选择。例如微分方程离散化后得到的系数矩阵中，非零元素仅占极小比例，这时就不适合使用矩阵分解的方法。这便引出了另一种思路——不直接求解，而是想办法逐步逼近最终答案，也就是本章主要介绍的迭代的基本思路。

**迭代法**是指从一个初始值出发，构造迭代格式逐步逼近最终解的求解方法。每次迭代过程中的计算主要是矩阵与向量的乘积，不需要像直接法那样显式分解矩阵，因此非常适合大型稀疏矩阵。

我们先来看迭代法的一个简单例子。将矩阵 $A$ 变形：$A=M-N$，其中 $M$ 为非奇异矩阵。因此 $Ax=b$ 变成了 $Mx=Nx+b$，从而可构造一个迭代格式：

$$
x^{(k+1)}=Bx^{(k)}+f,\quad k=0,1,2,\cdots
$$

其中 $B=M^{-1}N=M^{-1}(M-A)=I-M^{-1}A$，$f=M^{-1}b$ 被称为迭代矩阵。$M$ 的不同选取方式对应了几种不同的迭代法（如Jacobi、Gauss-Seidel、SOR等）。

这种迭代法收敛的充要条件是迭代矩阵的谱半径 $\rho(B)<1$，收敛一个充分条件是存在某种算子范数使得 $\|B\|<1$。由于 $\rho(B)\leqslant \|B\|$，范数条件更强，但更容易验证。渐近收敛速度定义为$R(B)=-\ln\rho(B).$ $\rho(B)$ 越小，$R(B)$ 越大，收敛越快。若要求误差压缩到初始误差的 $\sigma$ 倍，所需迭代次数约为$k\geqslant \frac{-\ln\sigma}{R(B)}.$

这一示例表明：**迭代法的核心问题是收敛性和收敛速度**，而这两者都和迭代矩阵的谱性质有关。

4.1节中，我们已经介绍过对称正定矩阵的重要性。对于直接法难以处理的对称正定线性方程组（即对称正定系统）问题，在众多迭代方法中，**共轭梯度法**（Conjugate Gradient，CG）是最有效的方法之一。和前面介绍的迭代法不同的是，它的基本思想是把求 $Ax=b$ 的解转化为求一个关于矩阵的二次函数最小值点的问题。当 $A$ 对称正定时，函数

$$
\phi(x)=\frac12 x^T A x-b^T x
$$

是一个严格凸的二次函数，其唯一最小值点恰好是 $Ax=b$ 的解。这个函数的等高线是一族同心椭圆（在高维空间中为椭球面），最小值点位于椭圆的中心。求解 $Ax=b$ 因此等价于在这个曲面上寻找最低点。因此可以用梯度下降的思想进行迭代法求解。

最直接的想法是沿负梯度方向搜索，每一步在当前方向上达到最优的**最速下降法**。迭代格式为：

$$
x_{k+1}=x_k+\alpha_k r_k,\quad k=1,2,\cdots
$$

其中 $r_k=b-Ax_k$ 为第 $k$ 步的残差，$\alpha_k$ 为步长。$\alpha_k$ 的取值为 $\phi(x_k+\alpha r_k)$ 取最小值时的对应值，可以求得 $\alpha_k=\frac{r_k^T r_k}{r_k^T A r_k}$。

最速下降法的收敛性估计为

$$
\|x-x_k\|_A\leqslant
\left(\frac{\kappa-1}{\kappa+1}\right)^k
\|x-x_0\|_A,
$$

其中 $\kappa$为谱条件数， $\|v\|_A=\sqrt{v^T A v}$。这个估计表明：$\kappa$ 越接近 $1$，收敛越快；$\kappa$ 很大时，收敛可能极慢。最速下降法的缺点是相邻两步的搜索方向正交，导致路径呈“锯齿状”，在某些病态问题中收敛极慢。

因此，**共轭梯度法**针对此问题作出改进：选择一组关于 $A$ 共轭的搜索方向，即满足 $p_i^T A p_j=0,\ i\neq j$。共轭方向避免了搜索方向的重复，可以在精确算术的前提下，使得一个 $n$ 维问题至多 $n$ 步即可收敛。从Krylov子空间的角度看，CG在第 $k$ 步于空间

$$
\mathcal{K}_k(A,r_0)=\operatorname{span}\{r_0,Ar_0,\dots,A^{k-1}r_0\}
$$

中寻找使误差在 $A$-范数意义下最小的近似解。

CG的适用条件明确：**$A$ 必须对称正定**。（若 $A$ 非对称或非正定，标准CG不适用，需要其他Krylov方法）。这种迭代方法的优点在于只需要计算矩阵和向量的乘法 $Ap$，且不需要显式存储 $A$，能有效地处理大规模的稀疏矩阵问题。

从几何直观角度理解，条件数 $\kappa(A)$ 决定了函数 $\phi(x)$ 所对应的等值面的形状。$\kappa$ 越接近 $1$，等高线越接近圆形，从任意点出发都能较快到达中心；$\kappa$ 越大，等高线越扁，形成“峡谷”状，迭代路径容易在峡谷两侧反复震荡，收敛缓慢。这正是4.1节末尾提到的条件数在迭代法中的直接体现。

解决了怎么迭代的问题之后，我们还需要知道应该在什么时候停止迭代，以及如何分析停止后的误差。**停止准则与解误差的关系**是本节需要澄清的一个重要概念。

在实际浮点运算中，由于舍入误差，CG的收敛行为不会像理论那样精确。CG的迭代过程自然产生残差 $r_k=b-Ax_k$。常用的停止准则是相对残差 $\frac{\|r_k\|}{\|b\|}$ 小于给定容差。然而，残差小并不意味着解误差小。由 $Ax=b$ 和 $Ax_k=r_k$ 做差，得 $A(x-x_k)=r_k$, 故误差与残差之间的关系是 $x-x_k=A^{-1}r_k$. 因此 $\|x-x_k\|\leqslant \|A^{-1}\|\,\|r_k\|$。又由 $b=Ax$ 得 $\|b\|\leqslant \|A\|\,\|x\|$. 可推出在相对误差的形式下，

$$
\frac{\|x-x_k\|}{\|x\|}\leqslant \|A^{-1}\|\,\|A\|\,\frac{\|r_k\|}{\|b\|}= \kappa(A)\frac{\|r_k\|}{\|b\|}.
$$

这表明残差到解误差的放大倍数，正是条件数。

在很多实际问题中，当 $\kappa(A)$ 很大时，即使相对残差很小，相对解误差仍可能很大。这与4.1节中条件数影响舍入误差放大倍数的现象十分相似：在这里，条件数影响着残差到解误差的放大倍数。在实际计算中，还可以用残差给出后验误差估计：

$$
\|x-x_k\|\leqslant \|A^{-1}\|\,\|r_k\|,
$$

其中 $\|A^{-1}\|$ 可以用条件数和 $\|A\|$ 估计，或者用其他方法近似。

由上述公式可知，当矩阵 $A$ 的条件数 $\kappa$ 很大时，CG的收敛速度会很慢。因此我们要进一步改进迭代方法，使得改进后的迭代格式的条件数尽可能地变小。**预条件**就是为加速CG收敛而出现的关键方法，选取一个矩阵 $M$ （被称作预条件子）将原系统 $Ax=b$ 转化为一个等价且条件数更小的系统：

$$
M^{-1}Ax=M^{-1}b,
$$

其中 $M$ 是对 $A$ 的某种近似，且 $M^{-1}$ 易于应用。这样处理的目标是使 $M^{-1}A$ 的条件数远小于 $A$ 的条件数，从而加速收敛。对这样使用预条件的系统应用共轭梯度法进行迭代，就是预条件共轭梯度法(简称PCG)。

这里$M$ 需要满足两个条件：一是对称正定（为了保持对称性）；二是 $M^{-1}$ 容易计算。 最简单的选择是对角预条件 $M=\operatorname{diag}(A)$，即取 $A$ 的对角线元素构成对角矩阵。为了保持对称性，有时也会写成

$$
M^{-\frac12}AM^{-\frac12}y=M^{-\frac12}b,\qquad y=M^{\frac12}x
$$

这个矩阵仍然对称正定。对角预条件计算代价极低（只需逐元素除法），但这种方法只利用了原矩阵的对角元素的信息，对病态性不主要体现在对角线上的问题来说，改善的效果相对有限。

其他类型的预条件子包括：

- **不完全Cholesky分解（IC）**：对 $A$ 做近似Cholesky分解，保留稀疏性，丢弃填充元。IC预条件对稀疏矩阵非常有效，但构造代价较高。
- **多重网格预条件**：利用问题的多尺度结构，构造层次化的预条件子。对由PDE离散化得到的矩阵特别有效。
- **稀疏近似逆（SPAI）**：直接构造 $M^{-1}$ 的稀疏近似，避免求解线性方程组。

（本章主要介绍对角预条件的相关内容，更多预条件的相关内容留到后续第8章学习。）

**一个实际运算的小例子。** 取二维对角正定系统

$$
A=\begin{bmatrix}1&0\\0&100\end{bmatrix},\qquad
b=\begin{bmatrix}1\\1\end{bmatrix}.
$$

真解为 $x_\star=(1,0.01)^T$，条件数 $\kappa_2(A)=100$。从零初值出发，标准CG的第一步为

$$
r_0=b=(1,1)^T,\quad p_0=r_0,
$$

$$
\alpha_0=\frac{r_0^Tr_0}{p_0^TAp_0}
=\frac{2}{1+100}=\frac{2}{101},
$$

$$
x_1=x_0+\alpha_0p_0
=\left(\frac{2}{101},\frac{2}{101}\right)^T
\approx(0.0198,0.0198)^T.
$$

第一个分量已接近真解 $1$ 的量级，但第二个分量 $0.0198$ 与真解 $0.01$ 还有差距。若改用对角预条件

$$
M=\operatorname{diag}(A)=\begin{bmatrix}1&0\\0&100\end{bmatrix},
$$

则预条件系统 $M^{-1}Ax=M^{-1}b$ 对应的矩阵为 $M^{-1}A=I$，条件数降为 $1$。PCG的第一步就等价于直接求解

$$
\widetilde{x}_1=M^{-1}b=(1,0.01)^T=x_\star.
$$

这个例子说明：对角预条件的作用正是把各个方向上的尺度差异“拉平”；当对角尺度差异是条件数的主要来源时，对角预条件的效果非常明显。但当病态来自非对角耦合时，仅靠对角预条件就不够，可能需要更强的预条件子。

要注意的是，预条件改善的是迭代矩阵的条件数，从而加快收敛，但它不改变原问题的病态性。PCG的总成本需要同时计入迭代次数的减少和每步迭代中应用预条件子的额外开销。在比较有无预条件的性能时，应报告迭代次数和总计算时间的对比，而不仅仅是迭代次数。

**易错点提示：** 残差小不一定代表着解误差小；迭代次数少不一定代表着总时间短。预条件子必须对称正定，否则PCG的对称性被破坏。此外，预条件改善的是收敛速度，不是问题的病态性；病态问题在预条件后仍然可能病态，只是迭代矩阵的条件数变小了。

### 关键推导：让结论有依据

**推导 1：从二次函数最小化到CG迭代格式。** 设 $A$ 对称正定，函数$\phi(x)=\frac12 x^T A x-b^T x$。其梯度为 $\nabla\phi(x)=Ax-b$，Hessian矩阵为 $A$，故最小值点满足 $Ax=b$。CG在第 $k$ 步沿方向 $p_k$ 搜索：

$$
x_{k+1}=x_k+\alpha_k p_k.
$$

步长 $\alpha_k$ 通过最小化 $\phi(x_k+\alpha p_k)$ 确定：

$$
\frac{d}{d\alpha}\phi(x_k+\alpha p_k)
=p_k^T(A(x_k+\alpha p_k)-b)=0,
$$

解得 $\alpha_k=\frac{p_k^T r_k}{p_k^T A p_k}$，其中 $r_k=b-Ax_k$ 为第 $k$ 步的残差。搜索方向 $p_k$ 由当前残差和上一方向组合而成：

$$
p_{k+1}=r_{k+1}+\beta_k p_k,
$$

其中 $\beta_k=\frac{r_{k+1}^T r_{k+1}}{r_k^T r_k}$。初始方向 $p_0=r_0$。残差递推公式为 $r_{k+1}=r_k-\alpha_k A p_k$。每一步只需计算一次矩阵与向量的乘积 $Ap_k$。对应基本迭代式：

$$
\alpha_k=\frac{r_k^Tr_k}{p_k^TAp_k},\quad
x_{k+1}=x_k+\alpha_kp_k,\quad
r_{k+1}=r_k-\alpha_kAp_k,
$$

$$
\beta_k=\frac{r_{k+1}^Tr_{k+1}}{r_k^Tr_k},\qquad
p_{k+1}=r_{k+1}+\beta_kp_k.
$$

从上述推导中可以看出，若 $A$ 非对称或非正定，$\phi(x)$ 不一定有最小值点，$p_k^TAp_k$ 也可能不为正，标准CG不适用。

设 $\kappa=\kappa_2(A)=\frac{\lambda_{\max}}{\lambda_{\min}}$。CG的误差在 $A$-范数意义下满足

$$
\|x-x_k\|_A\leqslant
2\left(\frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}\right)^k
\|x-x_0\|_A,
$$

其中 $\|v\|_A=\sqrt{v^T A v}$。这个估计表明：$\kappa$ 越接近 $1$，收敛越快；$\kappa$ 很大时，收敛可能极慢。估计式虽然通常偏保守（实际收敛往往更快，尤其当特征值聚簇时），但它清晰地揭示了条件数对收敛速度的支配作用。

**推导 2：对角PCG的格式。** 设 $M=\operatorname{diag}(A)$。预条件系统为 $M^{-1}Ax=M^{-1}b$。定义 $z_k=M^{-1}r_k$。PCG的迭代格式为：

$$
\alpha_k=\frac{r_k^T z_k}{p_k^T A p_k},\quad
x_{k+1}=x_k+\alpha_k p_k,\quad
r_{k+1}=r_k-\alpha_k A p_k,
$$

$$
z_{k+1}=M^{-1}r_{k+1},\quad
\beta_k=\frac{r_{k+1}^T z_{k+1}}{r_k^T z_k},\quad
p_{k+1}=z_{k+1}+\beta_k p_k.
$$

注意 $M^{-1}r$ 的计算只需逐元素除以对角元，代价为 $O(n)$。对应预条件的对称变换：

$$
\widetilde A=M^{-1/2}AM^{-1/2},\quad y=M^{1/2}x,\quad
\widetilde Ay=M^{-1/2}b.
$$

这里要求 $M$ 对称正定，否则 $M^{1/2}$ 与 $M^{-1/2}$ 不一定存在，PCG的对称性会被破坏。实现时不必显式形成 $M^{1/2}$，只需每步解 $Mz_k=r_k$；对角预条件取 $M=\operatorname{diag}(A)$，逐元素除法即可。

PCG等价于对变换后的系统 $\widetilde{A}=M^{-1/2}AM^{-1/2}$ 应用标准CG，其中 $\widetilde{A}$ 仍然对称正定。因此，PCG的收敛速度由 $\widetilde{A}$ 的条件数 $\kappa(\widetilde{A})=\kappa(M^{-1/2}AM^{-1/2})$ 支配。若 $M$ 是 $A$ 的良好近似，则 $\kappa(\widetilde{A})\ll \kappa(A)$，收敛速度显著提高。这一推导说明了为什么预条件子需要对称正定：只有这样才能保持变换后系统的对称性。

### 实验设计：先预测，再运行

实验任务：对同一正定系统比较CG与对角PCG，在相同停止标准下记录迭代次数和实际误差；观察最速下降法的锯齿现象；验证残差与解误差的关系。

| 要填写的项目 | 本组内容 |
| --- | --- |
| 要回答的问题与运行前预测 | **预测：** 对角PCG的迭代次数少于标准CG；改善程度取决于矩阵对角占优的程度。对条件数大的矩阵，预条件的收益更明显。但预条件每步增加的计算量很小（对角预条件），因此总时间也可能减少。最速下降法的收敛路径呈锯齿状，CG的收敛路径更平滑。残差小不一定解误差小，条件数越大，残差到解误差的放大越明显。 |
| 输入、参数、单位、数据来源与划分 | 输入：对称正定矩阵，例如二维 Poisson 方程五点差分格式生成的矩阵（条件数随网格加密而增大），或随机生成的对角占优对称正定矩阵。规模 $n=100$ 或 $n=400$。右端 $b$ 随机生成或取 $b=A\mathbf{1}$。另取一个病态矩阵（如 Hilbert 矩阵或条件数大的对角矩阵）用于验证残差与解误差的关系。数据来源：本组用固定种子生成。 |
| 独立参考解/基线及选择理由 | 参考解：用 `numpy.linalg.solve`（密集）或直接法求得的解作为“真解”。基线：标准 CG。 |
| 误差指标、容差及其依据 | 指标：相对残差 $\frac{\lVert r_k\rVert_2}{\lVert b\rVert_2}$ 的历史；达到容差（如 $10^{-6}$）所需的迭代次数；解误差 $\frac{\lVert x_k-x_{\text{true}}\rVert_2}{\lVert x_{\text{true}}\rVert_2}$; 总计算时间；条件数 $\kappa_2(A)$。容差依据：与 CG 收敛性理论中的停止准则一致。 |
| 环境、种子、设备与运行预算 | 环境：Python 3 + NumPy/SciPy。种子：本组固定。设备：个人电脑。预算：秒级到分钟级（取决于矩阵规模）。 |

【待填写】在本章 Notebook 的同号小节编写代码、亲自运行并保存输出。

以下代码仅作为起步骨架，完整实现、实际输出与运行记录以 [A组-实验.ipynb](A2组-实验.ipynb) 的 4.2 节为准。正文引用相应小节并解释结果；两处共有的公式、参数和结论须同步。

```python
import numpy as np

np.set_printoptions(precision=6, suppress=True)


def cg(A, b, x0=None, rtol=1e-10, atol=0.0, maxiter=None, M=None):
    """
    共轭梯度法（可选对角预条件）。

    参数
    ----
    A : (n, n) ndarray
        对称正定矩阵。
    b : (n,) ndarray
        右端向量。
    x0 : (n,) ndarray or None
        初始猜测；None 表示零向量。
    rtol, atol : float
        停止准则：||b - A x_k||_2 <= max(rtol * ||b||_2, atol)。
    maxiter : int or None
        最大迭代步数；None 表示 10 * n。
    M : (n, n) ndarray or None
        预条件矩阵，要求对称正定。None 表示不使用预条件；
        对角预条件时传入 np.diag(np.diag(A))。
        注意：SciPy cg 的 M 参数代表逆预条件作用 M^{-1}，
        与本节公式中的 M 不可按变量名直接混同。

    返回
    ----
    x : (n,) ndarray
        近似解。
    info : dict
        包含迭代次数、真实残差历史、是否收敛等信息。
    """
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    n = b.size

    if x0 is None:
        x = np.zeros(n)
    else:
        x = np.asarray(x0, dtype=float).copy()

    if maxiter is None:
        maxiter = 10 * n

    # 真实残差，而不是递推残差
    r = b - A @ x
    b_norm = np.linalg.norm(b, 2)
    tol = max(rtol * b_norm, atol)

    # 预条件作用：z = M^{-1} r
    if M is None:
        z = r.copy()
    else:
        # 对角预条件可以直接逐元素相除；
        # 一般 M 应通过求解 M z = r 实现。
        z = np.linalg.solve(M, r)

    p = z.copy()
    rz_old = float(r @ z)

    residual_history = [np.linalg.norm(r, 2)]
    converged = False

    for k in range(maxiter):
        if np.linalg.norm(r, 2) <= tol:
            converged = True
            break

        Ap = A @ p
        pAp = float(p @ Ap)
        if pAp <= 0:
            raise ValueError("p^T A p 非正，A 可能不是对称正定矩阵。")

        alpha = rz_old / pAp
        x = x + alpha * p
        r = r - alpha * Ap

        residual_history.append(np.linalg.norm(r, 2))

        if M is None:
            z = r.copy()
        else:
            z = np.linalg.solve(M, r)

        rz_new = float(r @ z)
        beta = rz_new / rz_old
        p = z + beta * p
        rz_old = rz_new

    info = {
        "iterations": k,
        "converged": converged,
        "residual_history": np.asarray(residual_history),
        "final_residual_norm": float(np.linalg.norm(r, 2)),
    }
    return x, info


def test_cg_vs_pcg():
    rng = np.random.default_rng(2026)
    n = 100

    # 构造对称正定矩阵：A = D T D，其中 T 为三对角 SPD，D 为正对角
    T = np.diag(3.0 * np.ones(n))
    T += np.diag(-1.0 * np.ones(n - 1), -1)
    T += np.diag(-1.0 * np.ones(n - 1), 1)

    d = np.exp(np.linspace(0.0, 4.0, n))  # 尺度差异
    D = np.diag(d)
    A = D @ T @ D
    A = 0.5 * (A + A.T)  # 数值对称化

    x_true = rng.normal(size=n)
    b = A @ x_true

    x_cg, info_cg = cg(A, b, rtol=1e-10, atol=0.0, maxiter=2000)
    x_pcg, info_pcg = cg(A, b, rtol=1e-10, atol=0.0, maxiter=2000,
                         M=np.diag(np.diag(A)))

    print("CG 迭代次数:", info_cg["iterations"])
    print("CG 最终相对残差:",
          info_cg["final_residual_norm"] / np.linalg.norm(b, 2))
    print("CG 相对真解误差:",
          np.linalg.norm(x_cg - x_true, 2) / np.linalg.norm(x_true, 2))

    print("PCG 迭代次数:", info_pcg["iterations"])
    print("PCG 最终相对残差:",
          info_pcg["final_residual_norm"] / np.linalg.norm(b, 2))
    print("PCG 相对真解误差:",
          np.linalg.norm(x_pcg - x_true, 2) / np.linalg.norm(x_true, 2))

    print("cond_2(A) =", np.linalg.cond(A, 2))


if __name__ == "__main__":
    test_cg_vs_pcg()

```

In [5]:
# 4.2 输入与独立参考
# TODO：填写本节输入、参数、种子和独立参考；不得用待验证算法自身输出冒充真值。
# 在这里补充代码；完成后删除这条占位提示。
print('4.2 输入与独立参考：未完成，请填写并实际运行。')


4.2 输入与独立参考：未完成，请填写并实际运行。


In [6]:
# 4.2 方法实现与运行
# TODO：对同一正定系统比较CG与对角预条件CG，在相同停止标准下记录迭代次数和实际误差。
# 在这里补充代码；完成后删除这条占位提示。
print('4.2 方法实现与运行：未完成，请填写并实际运行。')


4.2 方法实现与运行：未完成，请填写并实际运行。


In [7]:
# 4.2 误差图表与比较
# TODO：绘制相对残差历史，报告初值、容差、真解误差与预条件成本。
# 在这里补充代码；完成后删除这条占位提示。
print('4.2 误差图表与比较：未完成，请填写并实际运行。')


4.2 误差图表与比较：未完成，请填写并实际运行。


### 结果解释与本节结论

应提供的证据：绘制相对残差历史，报告初值、容差、真解误差与预条件成本；比较最速下降法与CG的收敛路径；验证残差与解误差的关系。


- **CG与PCG的迭代次数**：记录标准CG与对角PCG达到同一真实残差阈值所需的迭代次数；报告 $\kappa_2(A)$ 与 $\kappa_2(\widetilde A)$。若预条件后迭代次数明显减少，说明对角预条件改善了对角尺度差异；若改善有限，说明该矩阵的主要病态来源不是对角尺度，而可能来自非对角耦合。
- **残差历史**：绘制 $\frac{\|r_k\|_2}{\|b\|_2}$ 随迭代次数的变化曲线，对数纵轴。注意：递推残差与真实残差在有限精度下可能偏离，因此教学代码每步重新计算真实残差。图中需注明初值、容差、矩阵规模与条件数。
- **真解误差**：报告最终相对真解误差 $\frac{\|x_k-x_{\text{true}}\|_2}{\|x_{\text{true}}\|_2},$ 若最终相对残差很小但相对真解误差仍较大，则与式

  $$
  \frac{\|x-x_k\|}{\|x\|}\leqslant \kappa(A)\frac{\|r_k\|}{\|b\|}
  $$

  一致，说明条件数放大了残差到解误差的倍数。
- **预条件成本**：对角预条件的构造与应用成本很低，但仍应记录 `M` 的构造时间与应用时间。若比较总时间，应报告 $T_{\mathrm{CG}}$ 与 $T_{\mathrm{setup}}+T_{\mathrm{PCG}}$，而不是只比较迭代次数。
- **最速下降法与CG的路径**：对二维或三维小矩阵，可画出等高线与迭代路径，观察最速下降法相邻方向正交、路径呈锯齿状，而CG的路径更平滑。若未运行，写“未运行”。

**本节结论：** 迭代法的核心是收敛性和收敛速度，两者都由迭代矩阵的谱性质决定。CG适用于对称正定矩阵，其收敛速度由条件数支配。预条件通过改善迭代矩阵的条件数来加速收敛，但改善的是收敛速度，不是问题的病态性。残差小不一定解误差小，条件数决定了残差到解误差的放大倍数。

### 易错点与讨论

待核验的风险：残差小不一定解误差小；迭代次数少不一定总时间短。


1. **残差与解误差的区别**：残差是输出端的误差，解误差是输入端的误差。条件数描述了从输出端误差到输入端误差的放大倍数。小残差只在条件数不很大时才能保证小解误差。可用4.3节的开场对角矩阵 $A=\operatorname{diag}(1,10^{-10})$ 说明：对扰动后的右端残差为零，但相对原真解误差约为 $0.707107$。
2. **预条件改善的对象**：预条件改善的是迭代的收敛速度，不是解本身，也不是问题的病态性。预条件CG的总成本需要同时计入迭代次数的减少和每步迭代中应用预条件子的额外开销。对角预条件每步成本 $O(n)$，但不完全Cholesky等预条件的构造成本可能较高。
3. **CG的适用条件**：$A$ 必须对称正定。若 $A$ 非对称或非正定，标准CG不适用，需要其他Krylov方法。
4. **最速下降法与CG的区别**：最速下降法沿负梯度方向搜索，相邻方向正交，收敛慢；CG选择共轭方向，避免重复搜索，收敛快。

**随堂提问：** 设计一题比较有无预条件的收益，并要求计入构造与应用成本。设计一个病态矩阵，验证残差小但解误差大的现象。


- **明确条件**：给出一个对称正定矩阵 $A$，其条件数较大；再给出对角预条件矩阵 $M=\operatorname{diag}(A)$。要求分别用CG和PCG求解，并记录迭代次数、真实残差、真解误差、预条件构造时间与应用时间。
- **同学可能的回答**：一部分同学可能只比较迭代次数，认为PCG一定更快；另一部分可能忽略预条件构造成本。
- **本组最终解释**：PCG的收益应同时看迭代次数和总时间。对角预条件构造成本低，通常总时间也减少；但不完全Cholesky等预条件构造成本高，可能迭代次数减少很多，总时间却未必减少。因此必须报告 $T_{\mathrm{setup}}+T_{\mathrm{PCG}}$，而不是只报告迭代次数。


### 4.2 实际运行记录

【待填写】运行日期、完整环境/新增依赖版本、代码状态、输出位置、误差与结论。对照上面的证据要求逐项解释；未运行写未运行。AI辅助内容及人工改动记录在本章模板-A组-AI对话记录.md。

### 概念讲解：把问题说明白

4.1节和4.2节分别讨论了求解 $Ax=b$ 的直接法和迭代法。直接法关心如何稳定、高效地分解矩阵；迭代法关心收敛速度。这两节都反复出现了同一个概念：矩阵的条件数。直接法中，条件数决定了舍入误差的放大倍数；迭代法中，条件数支配了CG的收敛速度。那么，条件数究竟由矩阵的什么结构决定？矩阵的“敏感方向”和什么有关？如何用低秩矩阵近似一个给定矩阵？这些问题都需要从矩阵的谱结构来回答，这正是本节讨论的核心内容。

**特征值与特征向量**是理解矩阵作用的第一把钥匙。我们知道，对方阵 $A$，若存在数 $\lambda$ 和非零向量 $v$，使得$Av=\lambda v,$ 则称 $\lambda$ 为 $A$ 的特征值，$v$ 称为 $\lambda$ 的一个特征向量。

特征值可以用于判断矩阵是否奇异（有零特征值则奇异），也可以用于分析矩阵的稳定性（谱半径 $\rho(A)=\max_i|\lambda_i|$ 和迭代法的收敛性有关）。此外，特征值还有两个基本性质：

$$
\det(A)=\lambda_1\lambda_2\cdots\lambda_n,\qquad
\operatorname{tr}(A)=\lambda_1+\lambda_2+\cdots+\lambda_n.
$$

这两个性质分别联系了行列式、迹与特征值，在理论分析和实际计算中都十分有用。

特征值在矩阵的相关计算方面最常见的应用是求矩阵的幂次。如果方阵 $A$ 有 $n$ 个线性无关的特征向量，那么 $A$ 可以在这组基下对角化: $A=V\Lambda V^{-1}$。进一步，有 $A^n= V\Lambda^n V^{-1} $, 可以利用这个结果便捷地计算 $A^n$ ,在求解关于 $A$ 的微分方程，以及一些迭代计算中，这一方法非常重要。(不可对角化的方阵可以利用Jordan标准型进行类似计算，但不是本章重点，不再展开讲解)

特征值的几何含义是：矩阵 $A$ 作用在特征向量 $v$ 上，只改变其长度(伸缩因子为 $\lambda$)，不改变方向。若 $A$ 可对角化，那么$A$ 的作用可以看作完全由它在各个特征方向上的伸缩决定。因此，研究矩阵 $A$ 对一般向量的作用时，可以把 $A$ 的所有特征向量作为一组基。

进一步地，我们希望研究矩阵对向量作用的影响主要取决于什么，并通过研究矩阵对向量反复作用的结果来说明这一点。假设 $A$ 的特征值按模长大小排列为: $|\lambda_1|>|\lambda_2|\geqslant \cdots \geqslant|\lambda_n|,$ 其中 $\lambda_i$ 对应特征向量 $v_i (i=1,2,\dots,n),|\lambda_1|$ 严格最大。那么对任意初值 $x_0$ ,可以用 $A$ 的特征向量表示为

$$
x_0=c_1v_1+c_2v_2+ \cdots +c_nv_n.
$$

假设 $c_1\neq0$, 反复左乘 $A$, 得到：

$$
x_k=A^kx_0 =c_1\lambda_1^k v_1+c_2\lambda_2^k v_2+\cdots+c_n\lambda_n^k v_n.
$$

 $k$ 充分大时，$\lambda_1^k$ 相对来说远大于其他项，因此归一化后 $x_k$ 的方向趋近 $v_1$。这就是**幂迭代**的思想：反复用矩阵作用在向量上，最大特征值对应的方向会逐渐占据主导地位，因此也被称为主特征向量方向。用这种思路便得到了计算最大特征值的幂迭代算法：给定 $x_0$，进行如下的迭代：

$$
y_k=Ax_{k-1},\qquad x_k=\frac{y_k}{\|y_k\|}.
$$

则特征值可用Rayleigh商估计,第 $k$ 步得到的估计为：

$$
\lambda_1^{(k)}\approx \frac{x_k^TAx_k}{x_k^Tx_k}.
$$

需要强调这里的假设条件：最大模特征值 $\lambda_1$ 的模必须严格最大；初值 $x_0$ 在主特征向量方向上的展开系数 $c_1$ 不能为零；矩阵最好可对角化；负主特征值可能造成符号交替。若主导模并列、矩阵有缺陷或初值投影为零，简单幂迭代不适用。

尽管特征值是一个对于研究矩阵非常重要的概念，特征值分解也还是有两个重要局限：其一，它只适用于方阵；其二，即使对于方阵，也并非所有矩阵都能对角化。因此，我们需要在一般的矩阵上寻找一个具有类似概念和作用的值，并找到一种分解，它对任意矩阵都存在，且像特征值分解一样，能揭示矩阵在若干方向上的伸缩作用的效果。**奇异值分解**（Singular Value Decomposition，SVD）克服了特征值的上述局限：对任意 $m\times n$ 实矩阵 $A$，都存在分解:

$$
A=U\Sigma V^T,
$$

其中 $U$ 为 $m$ 阶正交矩阵，$V$ 为 $n$ 阶正交矩阵，$\Sigma$ 为 $m\times n$ 阶矩形对角矩阵，其对角元 $\sigma_1 \geqslant \sigma_2 \geqslant \cdots \geqslant \sigma_{\min(m,n)} \geqslant 0$,  称为奇异值。这一分解对任意矩阵都存在，不要求方阵、可逆或对称。正交矩阵 $U$ 和 $V$ 的列向量分别称为左奇异向量和右奇异向量。

SVD的几何含义是：任意矩阵 $A$ 的作用可以分解为三步——先用 $V^T$ 旋转（或反射）输入空间，再用 $\Sigma$ 沿坐标轴伸缩，最后用 $U$ 旋转输出空间。因此，若 $A$ 把单位球映射为一个椭球，则椭球的半轴长度就对应着矩阵的奇异值。

奇异值与特征值之间有明确的联系。对矩阵 $A=U\Sigma V^T,$ 考虑  $A^TA=V \Sigma^T \Sigma V^T$ ，是一个对称半正定矩阵，且易知其特征值为 $\sigma_i^2$，特征向量为 $V$ 的列向量；同理， $AA^T=U\Sigma\Sigma^TU^T$ 的特征值为 $\sigma_i^2$，特征向量为 $U$ 的列向量。因此，奇异值可以通过求出 $A^TA$ 的特征值再开方得到。

不过，数值计算中通常不直接计算 $A^TA$ 的特征值，因为这样会很容易损失精度：$A^TA$ 的条件数是 $A$ 的条件数的平方，舍入误差会被放大。而是使用专门的SVD算法。实际使用的SVD算法一般是直接对 $A$ 做正交变换，避免形成 $A^TA$。但 $A^TA$ 的特征值分解在理论分析中仍然重要，它建立了SVD与对称矩阵特征值分解的联系。这一联系为SVD的数值计算提供了理论基础。更重要的是，奇异值总为非负实数，且左、右奇异向量各自构成正交系，这比特征向量（可能复数、可能不正交）具有更良好的性质。

幂迭代方法的目标是寻找方阵最大特征值的方向，而SVD可看作把“找主方向”的思想推广到任意矩阵：右奇异向量 $v_i$ 是 $A^TA$ 的特征向量，左奇异向量 $u_i$ 是 $AA^T$ 的特征向量，奇异值 $\sigma_i$ 是 $A^TA$ 特征值的平方根。因此SVD同时给出输入空间和输出空间的一对主方向，比特征值分解更通用。

秩与奇异值之间有直接的关系。设矩阵 $A$ 的秩 $r=\operatorname{rank}(A)$, 则 $A$ 非零奇异值的个数即为 $r$. 若从数值计算角度出发，当某些奇异值极小时，数值运算中会被视为 $0$, 因此矩阵的“数值秩”可能小于理论秩。

条件数与奇异值之间也有紧密的关系。设 $A=U\Sigma V^T$, 可逆,则$A^{-1}=V\Sigma^{-1}U^T,$ 奇异值为 $\frac{1}{\sigma_i}(i=1,2,\cdots,n)$. 因此

$$
\|A\|_2=\sigma_1,\ \|A^{-1}\|_2=\frac{1}{\sigma_{\min}},\ \kappa_2(A)=\|A\|_2\|A^{-1}\|_2=\frac{\sigma_1}{\sigma_{\min}}.
$$

特别地，当 $A$ 为对称方阵时，$\kappa_2(A)=\frac{\max\limits_{1\leqslant i \leqslant n}|\lambda_i|}{\min\limits_{1\leqslant i \leqslant n}|\lambda_i|}.$

这里给出条件数的一些基本性质，在处理相关问题时很有用：

1. 对任何非奇异矩阵 $A$，都有 $\kappa(A)\geqslant 1$；
2. 对任何非零常数 $c$，$\kappa(cA)=\kappa(A)$；
3. 若 $Q$ 为正交矩阵，则 $\kappa(QA)=\kappa(AQ)=\kappa(A)$。

性质3说明，正交变换不改变条件数，这也是SVD数值稳定的原因之一：SVD只使用正交变换。

**奇异值分解与敏感性**的联系是本节的核心。考虑 $Ax=b$，$A$ 可逆。若右端有扰动 $\delta b$，解的变化满足 $A\delta x=\delta b$，即 $\delta x=A^{-1}\delta b$。将SVD代入，$A^{-1}=V\Sigma^{-1}U^T$。扰动量 $\delta b$ 若在左奇异向量 $u_i$ 方向上有分量，则解中该分量被放大 $\frac{1}{\sigma_i}$ 倍。特别地，沿最小奇异值 $\sigma_{\min}$ 对应的左奇异向量方向上的扰动会被放大最多 $\frac{1}{\sigma_{\min}}$倍，这是最坏情况下的放大。因此，最小的奇异值对应着最敏感的方向。

当 $\sigma_{\min}$ 非常小时，$A$ 接近奇异，矩阵的作用在某个方向上几乎把向量压缩为零，因此逆运算在该方向上的放大极大。这解释了开场例子中 $A=\operatorname{diag}(1,10^{-10})$ 的行为：最小奇异值为 $10^{-10}$，其对应方向上的扰动被放大 $10^{10}$ 倍，条件数正是这一放大倍数的上界。

设 $A$ 可对角化，$A=V\Lambda V^{-1}$。若 $A$ 有扰动 $\delta A$，则特征值扰动满足

$$
|\lambda_i(A+\delta A)-\lambda_i(A)|
\le\|V\|\,\|V^{-1}\|\,\|\delta A\|_2.
$$

这里$\|V\|\,\|V^{-1}\|$ 是特征向量矩阵的条件数。对对称矩阵，$V$ 正交，$\|V\|\,\|V^{-1}\|=1$，特征值对扰动不敏感；对非对称矩阵，特征向量矩阵可能病态，特征值对扰动敏感。这进一步说明SVD的优势：奇异值对扰动不敏感，因为SVD只使用正交变换。

类似地，若系数矩阵本身有扰动 $\delta A$，解的相对误差上界为

$$
\frac{\|\delta x\|}{\|x\|}\leqslant \kappa_2(A)\frac{\|\delta A\|}{\|A\|}.
$$

这一估计式与4.2节中残差与解误差的关系式形式一致，条件数总是出现在右端作为放大因子。因此，条件数不仅决定迭代法的收敛速度，也决定问题对输入扰动的敏感程度。它是矩阵结构的一个内在量，与具体的算法无关。

**截断奇异值分解与低秩近似**是SVD的另一个核心应用。将SVD按列写开，写成外积形式：

$$
A=\sum_{i=1}^{r}\sigma_i u_i v_i^T,
$$

其中 $r=\operatorname{rank}(A)$。每一项是一个秩为 $1$ 的矩阵，$\sigma_i$ 给出该项的“权重”。保留前 $k$ 个最大的奇异值及其对应的奇异向量，得到

$$
A_k=\sum_{i=1}^{k}\sigma_i u_i v_i^T.
$$

$A_k$ 的秩不超过 $k$，称为截断SVD（或低秩近似）。Eckart–Young–Mirsky定理指出：在Frobenius范数和谱范数下，$A_k$ 是所有秩不超过 $k$ 的矩阵中与 $A$ 最接近的。误差分别为

$$
\|A-A_k\|_F^2=\sum_{i=k+1}^{r}\sigma_i^2,
\qquad
\|A-A_k\|_2=\sigma_{k+1}.
$$

这一结论说明，奇异值的衰减速度决定了低秩近似的效果：若奇异值迅速衰减，则较小的 $k$ 就能捕捉 $A$ 的主要结构；若奇异值衰减缓慢，则需要较大的 $k$ 才能达到相同的误差。这为4.4节中的PCA和LoRA提供了理论基础：PCA用截断SVD对中心化数据降维，LoRA用低秩增量近似参数更新，二者的有效性都依赖于奇异值（或相应结构）的衰减假设。

**易错点提示：** 非对称矩阵的特征值绝对值一般不等于奇异值；把低秩重构等同于预测，是错误的。低秩近似是在Frobenius或谱范数意义下的最佳逼近，但不代表它能准确预测未知数据或泛化到分布之外。此外，数值计算中不应直接计算 $A^TA$ 的特征值来求奇异值，因为这会损失精度。

### 关键推导：让结论有依据

**推导 1：手算对角矩阵的特征值与奇异值。** 取

$$
A=\operatorname{diag}(1,10^{-10}).
$$

特征值：对角矩阵的特征值就是对角元，因此

$$
\lambda_1=1,\qquad \lambda_2=10^{-10}.
$$

奇异值：对角矩阵的奇异值等于对角元的绝对值，因此

$$
\sigma_1=1,\qquad \sigma_2=10^{-10}.
$$

对这个对角矩阵，特征值绝对值等于奇异值。条件数

$$
\kappa_2(A)=\frac{\sigma_1}{\sigma_2}=10^{10}.
$$

右端第二分量增加 $10^{-10}$ 后，精确解为 $\widetilde x=(1,2)^T$，而原真解为 $x_\star=(1,1)^T$，因此

$$
\|\widetilde x-x_\star\|_2=1,\qquad
\frac{\|\widetilde x-x_\star\|_2}{\|x_\star\|_2}=\frac{1}{\sqrt2}\approx0.707107.
$$

对原右端的相对残差为 $10^{-10}$，对扰动后右端的残差为0。因此，小残差与小解误差是不同主张，残差针对哪个右端必须说明。

**推导 2：一般矩阵特征值与奇异值不等同。** 取

$$
J=\begin{bmatrix}1&10\\0&1\end{bmatrix}.
$$

$J$ 是上三角矩阵，特征值为主对角元，因此两个特征值均为1。计算

$$
J^TJ=\begin{bmatrix}1&10\\10&101\end{bmatrix}.
$$

其特征值为

$$
51\pm10\sqrt{26},
$$

因此 $J$ 的奇异值约为

$$
\sigma_1\approx10.09902,\qquad \sigma_2\approx0.09902.
$$

显然 $J$ 的奇异值不等于其特征值的绝对值。实对称矩阵确有“奇异值等于特征值绝对值”的性质，更一般的正规矩阵也有相应结论，但不能推广到任意非对称矩阵。

**推导 3：截断SVD误差与范数。** 设

$$
A_k=\sum_{i=1}^k\sigma_i u_i v_i^T.
$$

则

$$
A-A_k=\sum_{i=k+1}^r\sigma_i u_i v_i^T.
$$

由于 $\{u_i\}$ 和 $\{v_i\}$ 均为正交系，

$$
\|A-A_k\|_F^2=\sum_{i=k+1}^r\sigma_i^2,\qquad
\|A-A_k\|_2=\sigma_{k+1}.
$$

在秩不超过 $k$ 的矩阵集合中，$A_k$ 使上述两个范数同时达到最小。这就是Eckart–Young–Mirsky定理。证明思路：对任意秩不超过 $k$ 的矩阵 $B$，在 $\operatorname{span}\{v_1,\dots,v_{k+1}\}$ 内存在单位向量 $z$ 使 $Bz=0$，于是

$$
\|A-B\|_2\ge\|(A-B)z\|_2=\|Az\|_2\ge\sigma_{k+1}.
$$

而 $A_k$ 恰好达到该下界。Frobenius范数的证明类似，利用正交投影和尾部平方和。

### 实验设计：先预测，再运行

实验任务：运行现有敏感性例子，再对小矩阵做不同秩截断，比较保留奇异值与重构误差。

| 要填写的项目 | 本组内容 |
| --- | --- |
| 要回答的问题与运行前预测 | **问题：** 1）对角矩阵 $A=\operatorname{diag}(1,10^{-10})$ 的条件数与扰动放大是否一致？2）一般非对称矩阵的特征值绝对值是否等于奇异值？3）截断SVD的重构误差是否随秩 $k$ 增加单调下降？4）奇异值衰减速度如何影响低秩近似效果？**预测：** 对角矩阵最小奇异值为 $10^{-10}$，条件数为 $10^{10}$，扰动在对应方向被放大 $10^{10}$ 倍；非对称矩阵 $J=\begin{bmatrix}1&10\\0&1\end{bmatrix}$ 的特征值全为1，而奇异值约为10.09902和0.09902，两者不等同；截断SVD的重构误差随 $k$ 增加而下降；奇异值衰减快时，较小 $k$ 就能达到很小误差。 |
| 输入、参数、单位、数据来源与划分 | 输入：对角矩阵 $A_1=\operatorname{diag}(1,10^{-10})$；非对称矩阵 $J=\begin{bmatrix}1&10\\0&1\end{bmatrix}$；随机矩阵 $A_2$（$4\times4$）；低秩矩阵 $A_3=UV^T$（$U,V$ 为随机矩阵，秩为2）；病态矩阵 $A_5$（如Hilbert矩阵或条件数大的对角矩阵）。数据来源：本组用固定种子生成。 |
| 独立参考解/基线及选择理由 | 参考解：直接用 `numpy.linalg.svd` 计算奇异值和奇异向量；重构误差的理论值由公式 $\sqrt{\sum\limits_{i=k+1}^r\sigma_i^2}$ 给出。基线：完整SVD（$k=r$）。 |
| 误差指标、容差及其依据 | 指标：奇异值谱；截断SVD的重构误差 $\|A-A_k\|_F$ 和 $\|A-A_k\|_2$；理论误差与实际误差的对比；特征值与奇异值的对比；条件数 $\kappa_2(A)$；扰动放大倍数。容差：约 $10^{-12}$ 量级。 |
| 环境、种子、设备与运行预算 | 环境：Python 3 + NumPy。种子：本组固定。设备：个人电脑。预算：秒级。 |

【待填写】在本章Notebook的同号小节编写代码、亲自运行并保存输出。

```python
import numpy as np

np.set_printoptions(precision=6, suppress=True)

# 1. 对角矩阵：条件数与扰动放大
A1 = np.diag([1.0, 1e-10])
b1 = A1 @ np.array([1.0, 1.0])
db1 = np.array([0.0, 1e-10])
x_tilde = np.linalg.solve(A1, b1 + db1)
x_star = np.array([1.0, 1.0])
print("cond_2(A1) =", np.linalg.cond(A1, 2))
print("x_tilde =", x_tilde)
print("relative solution error =",
      np.linalg.norm(x_tilde - x_star, 2) / np.linalg.norm(x_star, 2))
print("residual w.r.t. original b =",
      np.linalg.norm(b1 - A1 @ x_tilde, 2) / np.linalg.norm(b1, 2))
print("residual w.r.t. perturbed b =",
      np.linalg.norm((b1 + db1) - A1 @ x_tilde, 2) / np.linalg.norm(b1 + db1, 2))

# 2. 非对称矩阵：特征值与奇异值不等同
J = np.array([[1.0, 10.0],
              [0.0, 1.0]])
eig_J = np.linalg.eigvals(J)
sv_J = np.linalg.svd(J, compute_uv=False)
print("eigenvalues of J =", eig_J)
print("singular values of J =", sv_J)

# 3. 指定奇异值的矩阵与截断SVD
rng = np.random.default_rng(2026)
m, n = 8, 6
U, _ = np.linalg.qr(rng.normal(size=(m, m)))
V, _ = np.linalg.qr(rng.normal(size=(n, n)))
sigma = np.array([8.0, 4.0, 2.0, 1.0, 0.5, 0.1])
A2 = U[:, :n] @ np.diag(sigma) @ V.T

print("specified singular values =", sigma)
print("computed singular values  =", np.linalg.svd(A2, compute_uv=False))

for k in range(0, n + 1):
    if k == 0:
        A_k = np.zeros_like(A2)
    else:
        Uk, sk, Vtk = np.linalg.svd(A2, full_matrices=False)
        A_k = Uk[:, :k] @ np.diag(sk[:k]) @ Vtk[:k, :]
    err_2 = np.linalg.norm(A2 - A_k, 2)
    err_F = np.linalg.norm(A2 - A_k, 'fro')
    if k < n:
        theory_2 = sigma[k]
        theory_F = np.sqrt(np.sum(sigma[k:] ** 2))
    else:
        theory_2 = 0.0
        theory_F = 0.0
    print(f"k={k}: ||A-A_k||_2={err_2:.6e} (theory {theory_2:.6e}), "
          f"||A-A_k||_F={err_F:.6e} (theory {theory_F:.6e})")

# 4. 病态矩阵：条件数与敏感性
H = np.array([[1.0 / (i + j + 1) for j in range(5)] for i in range(5)])
print("cond_2(Hilbert_5) =", np.linalg.cond(H, 2))
```


### 已有起步实验：先运行并解释

小残差与大解误差的对角矩阵示例用于联系奇异值和条件数；不代表已完成LU、CG或AI应用。

以下保留原有代码供阅读、修改和复现。它只是本节的一个基线；请继续完成下方空白区。

In [8]:
import numpy as np

A = np.diag([1.0, 1e-10])
x_true = np.ones(2)
b = A @ x_true
delta_b = np.array([0.0, 1e-10])
x_perturbed = np.linalg.solve(A, b + delta_b)
relative_data_change = np.linalg.norm(delta_b) / np.linalg.norm(b)
relative_solution_change = np.linalg.norm(x_perturbed-x_true) / np.linalg.norm(x_true)
residual = np.linalg.norm(A @ x_perturbed - (b + delta_b))
np.testing.assert_allclose(x_perturbed, [1.0, 2.0])
assert residual < 1e-15
assert relative_solution_change > 0.7
assert np.linalg.cond(A) > 1e9
print({"condition_number": np.linalg.cond(A), "relative_data_change": relative_data_change,
       "relative_solution_change": relative_solution_change, "residual": residual})


{'condition_number': 10000000000.0, 'relative_data_change': 1e-10, 'relative_solution_change': 0.7071067811865475, 'residual': 0.0}


In [9]:
# 4.3 输入与独立参考
# TODO：填写本节输入、参数、种子和独立参考；不得用待验证算法自身输出冒充真值。
# 在这里补充代码；完成后删除这条占位提示。
print('4.3 输入与独立参考：未完成，请填写并实际运行。')


4.3 输入与独立参考：未完成，请填写并实际运行。


In [10]:
# 4.3 方法实现与运行
# TODO：运行现有敏感性例子，再对小矩阵做不同秩截断，比较保留奇异值与重构误差。
# 在这里补充代码；完成后删除这条占位提示。
print('4.3 方法实现与运行：未完成，请填写并实际运行。')


4.3 方法实现与运行：未完成，请填写并实际运行。


In [11]:
# 4.3 误差图表与比较
# TODO：给出谱/奇异值图、秩与误差表，分别解释扰动放大和信息压缩。
# 在这里补充代码；完成后删除这条占位提示。
print('4.3 误差图表与比较：未完成，请填写并实际运行。')


4.3 误差图表与比较：未完成，请填写并实际运行。


### 结果解释与本节结论

应提供的证据：给出谱/奇异值图、秩与误差表，分别解释扰动放大和信息压缩。


- **对角矩阵敏感性**：$A=\operatorname{diag}(1,10^{-10})$ 的最小奇异值为 $10^{-10}$，条件数为 $10^{10}$；右端第二分量增加 $10^{-10}$ 后，解的第二分量增加1，相对解误差约0.707107，与条件数上界一致。对原右端的相对残差约 $10^{-10}$，对扰动后右端的残差为0。小残差与小解误差是不同主张，需说明残差针对哪个右端。
- **特征值与奇异值对比**：$J=\begin{bmatrix}1&10\\0&1\end{bmatrix}$ 的特征值全为1，奇异值约为10.09902和0.09902，两者不等同。实对称矩阵和正规矩阵有特殊结论，但不能推广到任意非对称矩阵。
- **截断SVD重构误差**：对指定奇异值的矩阵，逐一检查 $r=0$ 至 $r=q$ 的重构误差。谱范数误差等于 $\sigma_{r+1}$，Frobenius误差等于尾部平方和开方。全秩重构的舍入残差保留在表中，不画成真正的零。
- **奇异值衰减与低秩近似**：若奇异值迅速衰减，较小 $k$ 就能捕捉主要结构；若衰减缓慢，需要较大 $k$ 才能达到相同误差。

**本节结论：** 特征值分解和SVD是理解矩阵结构的两种工具。特征值适用于方阵，描述矩阵在特定方向上的伸缩；SVD适用于任意矩阵，描述矩阵在所有方向上的伸缩，并给出秩、条件数、低秩近似的统一刻画。条件数由奇异值的比值定义，决定了问题对扰动的敏感程度。截断SVD给出了最佳低秩近似，其误差由被丢弃的奇异值决定。SVD的数值稳定性来自正交变换，这使它成为矩阵计算中最可靠的工具之一。

### 易错点与讨论

待核验的风险：把非对称矩阵的特征值绝对值普遍当作奇异值，或把低秩重构等同于预测。


**补充讨论：**

1. **特征值与奇异值的区别**：特征值只适用于方阵，奇异值适用于任意矩阵；非对称矩阵的特征值绝对值一般不等于奇异值；特征向量可能不正交，奇异向量总是正交的。反例：$J=\begin{bmatrix}1&10\\0&1\end{bmatrix}$，特征值全为1，奇异值约为10.09902和0.09902。
2. **低秩近似与预测的区别**：低秩近似是在Frobenius或谱范数意义下的最佳逼近，但不代表能准确预测未知数据或泛化到分布之外。低秩假设的有效性依赖于数据的低秩结构。
3. **数值计算中的SVD**：不应直接计算 $A^TA$ 的特征值来求奇异值，因为这会损失精度。实际使用的SVD算法直接对 $A$ 做正交变换。
4. **条件数与敏感性的关系**：最小奇异值对应最敏感方向，条件数是扰动放大倍数的上界。病态问题对扰动敏感，这是问题本身的固有性质，与算法无关。
5. **幂迭代的适用条件**：幂迭代要求主导模特征值严格最大、初值在主特征方向展开系数不为零、矩阵最好可对角化。若主导模并列、矩阵有缺陷或初值投影为零，简单幂迭代不适用。

**随堂提问：** 设计一个区分特征值与奇异值的反例题。设计一个矩阵，验证条件数与扰动放大的关系。

### 4.3 实际运行记录

【待填写】运行日期、完整环境/新增依赖版本、代码状态、输出位置、误差与结论。对照上面的证据要求逐项解释；未运行写未运行。AI辅助内容及人工改动记录在本章模板-A组-AI对话记录.md。

### 概念讲解：把问题说明白

4.1至4.3节建立了求解线性方程组、迭代法收敛性、特征值与SVD的理论。这些概念在机器学习中反复出现。本节的目标不是引入新的数值算法，而是说明这些矩阵方法如何嵌入学习模型，以及它们的适用边界在哪里。

如果把4.1-4.3看作“矩阵方法的三种基本操作”，那么4.4就是问：**这些操作在AI的哪些环节出场？出场时带来了什么新的问题？** 本节用一条统一的线索串联三块内容：**矩阵方法在AI中的三种典型嵌入方式**。

- **可微求解器**：矩阵求解被嵌入**训练循环**中，需要反向传播；
- **PCA**：矩阵分解被用于**数据预处理和降维**，不涉及训练；
- **LoRA**：矩阵分解被用于**参数高效微调**，是一种训练策略。

而在这三种场景中，4.1-4.3反复出现的**条件数**以三种不同的形式再次出场。这将是本节的贯穿线索。

#### 可微求解器：从“自动微分穿透”到“伴随方法”

**动机。** 深度学习框架依赖反向传播。若前向传播中包含一步线性求解，例如

$$
x^*=A^{-1}b,
$$

那么训练时需要对 $A$ 或 $b$ 求梯度。最直接的做法是让自动微分“穿透”求解过程。这就是所谓的 **“离散-优化”（discretize-then-optimize）** 思路：求解器内部执行的加法、乘法等常规算术运算，每一种都是可微的；既然求解操作是可微操作的组合，它本身也就是可微的。

**离散-优化的优点**是梯度精确（相对于实际使用的离散模型）、实现简单（只要求解器用PyTorch/JAX写就能自动求导）。**缺点**是内存效率低：如果求解器迭代了 $T$ 步，每步的内存成本为 $H$，那么总内存消耗为 $O(HT)$。

**伴随方法**的意义正在于此：它**不记录前向的每一步**，只利用前向分解的结果再解一次转置系统，把内存降到 $O(H)$。设损失 $\mathcal{L}$ 对解 $x^*$ 的梯度为 $v=\frac{\partial\mathcal{L}}{\partial x^*}$，则反向传播可以归结为一次伴随线性求解

$$
A^T\lambda=v,
$$

其中 $\lambda$ 是拉格朗日乘子。由 $Ax^*=b$ 对 $A$ 和 $b$ 微分可得

$$
\frac{\partial\mathcal{L}}{\partial b}=\lambda,\qquad
\frac{\partial\mathcal{L}}{\partial A}=-\lambda (x^*)^T.
$$

这一推导说明：可微求解器的反向传播不需要重新分解 $A$，而只需要利用前向分解的结果再解一次转置系统。当 $A$ 病态时，$A^T\lambda=v$ 同样病态，伴随求解的误差也会被放大。条件数在梯度计算中仍然起支配作用：$\kappa(A)$ 越大，梯度越不稳定。这正是4.3节条件数概念在训练循环中的延续。

实践中，深度学习框架的 `linalg.solve` 已经将这一伴随求解内置到反向传播中，避免了用户手动推导；Jaxopt等库则把优化层和线性求解器封装为可微模块。

#### PCA：中心化的必要性

**动机。** 先看一个失败案例。假设你收集了100个城市的“人口”和“GDP”数据。所有城市的人口和GDP都是正数，且总量都在增长。你对原始数据做PCA，发现第一主成分解释了99%的方差。你可能会兴奋地说“人口和GDP高度相关！”但真相是：**第一主成分只是捕获了“所有数据都是正数且远离原点”这个事实**——它指向的是“平均城市”的方向，而不是“人口变化与GDP变化的关系”。中心化后，第一主成分才真正反映“偏离平均值的协同变化”。

**PCA的目标。** 给定数据矩阵 $X\in\mathbb{R}^{n\times d}$（$n$ 个样本，$d$ 个特征），PCA的目标是找到一组正交方向，使得数据在这些方向上的投影方差最大。若数据已经中心化（每列均值为零），则协方差矩阵为

$$
C=\frac{1}{n-1}X^TX.
$$

$C$ 的特征向量即主成分方向，特征值 $\lambda_i$ 对应方差。对中心化矩阵 $X_c$ 做SVD：

$$
X_c=U\Sigma V^T,
$$

则 $V$ 的列向量为主成分方向，$U\Sigma$ 的列为得分，奇异值的平方与方差成比例：

$$
\lambda_i=\frac{\sigma_i^2}{n-1}.
$$

截断SVD保留前 $k$ 个主成分，解释方差比例为

$$
\frac{\sum\limits_{i=1}^k \sigma_i^2}{\sum\limits_{i=1}^r \sigma_i^2}.
$$

**中心化的必要性。** 若数据未中心化，第一主成分方向可能主要反映均值方向而非变异方向。更精确地说，未中心化时平方和可以分解为

$$
s_{tot}^2 = s_{mean}^2 + s_{err}^2,
$$

其中 $s_{mean}^2$ 是均值贡献的平方和，$s_{err}^2$ 是围绕均值的波动贡献的平方和。中心化后PCA分析的是 $s_{err}^2$，而未中心化时PC1会同时捕获 $s_{mean}^2$ 和 $s_{err}^2$。当数据的均值远离原点时（例如所有点都在第一象限），PC1会“指向”均值方向，而不是数据真正散布的方向。一个简单的二维例子：数据点聚集在 $(10,10)$ 附近但沿 $(1,-1)$ 方向散布，未中心化时PC1会指向 $(1,1)$ 方向（均值方向），中心化后PC1才指向 $(1,-1)$ 方向（真正的变异方向）。因此，默认情况下PCA应包含中心化步骤；对于某些特定场景（如光谱数据，需要保持基线信息），不中心化可能是合理的，但需要明确说明。

#### LoRA：低秩适配

**动机。** LoRA的核心思想来自一个经验观察：大模型在微调时，权重更新矩阵 $\Delta W$ 具有较低的内在秩。若预训练权重为 $W_0\in\mathbb{R}^{d\times k}$，全量微调更新为 $W_0+\Delta W$。LoRA不直接学习 $\Delta W$，而是将其参数化为低秩乘积

$$
\Delta W=BA,
$$

其中 $B\in\mathbb{R}^{d\times r}$，$A\in\mathbb{R}^{r\times k}$，$r\ll\min(d,k)$。前向传播变为

$$
h=W_0x+\frac{\alpha}{r}BAx.
$$

参数数量从 $dk$ 降为 $r(d+k)$。当 $r$ 远小于 $d$ 和 $k$ 时，参数量显著减少。

**可训练参数放在 $A$ 还是 $B$？** 这是一个值得追问的问题。

- 放在 $A$ 里（输入侧）：$A$ 把输入压缩到低维空间，$B$ 再从低维空间恢复。如果 $A$ 随机初始化、$B=0$，训练开始时 $BA=0$，模型行为不变。
- 放在 $B$ 里（输出侧）：$B$ 把低维空间映射回输出空间，$A=0$ 时同样 $BA=0$。

两种选择都“从预训练出发”，但训练动力学不同。LoRA原始论文中 $A$ 随机初始化、$B$ 零初始化（即Init[A]方案）。后续研究发现，**反过来初始化（$B$随机、$A$零初始化，即Init[B]）会导致训练动力学完全不同**：Init[A]允许使用更大的学习率，且会产生一种“内部不稳定性”——特征 $Az$ 很大但LoRA输出 $BAz$ 很小——这种不稳定性反而**促进了特征学习**；Init[B]则不会产生这种不稳定性，但训练是**次优的**。更进一步的后续研究甚至表明：**非零初始化（$A$和$B$都非零）也是可行的**，只要初始化方差合适，不会损害微调精度。这挑战了“必须从预训练模型严格出发”的传统观念，也说明初始化不是“随便选一个为零就行”，它影响训练效率。

缩放因子 $\frac{\alpha}{r}$ 控制低秩旁路的影响强度。

**LoRA与截断SVD的区别。** PiSSA等变体确实用截断SVD初始化LoRA：对预训练权重做SVD，取前 $r$ 个奇异值和奇异向量，令 $A=\sqrt{\Sigma_r}V_r^T$，$B=U_r\sqrt{\Sigma_r}$，并将残差保留在冻结权重中。但标准LoRA的 $A$ 和 $B$ 是通过训练学到的，初始时 $\Delta W=0$，它与“对 $W_0$ 直接做截断SVD”是两件事。后者是对已有权重的压缩或近似，前者是学习一个增量。

**易错点提示：** LoRA不等于对预训练权重直接做截断SVD；PCA未中心化会改变主成分的解释。可微求解器的梯度计算受条件数影响，并非无条件稳定。

### 关键推导：让结论有依据

**推导 1：线性求解的伴随方法。** 设 $x^*$ 满足 $Ax^*=b$，损失 $\mathcal{L}$ 通过 $x^*$ 依赖 $A$ 和 $b$。定义拉格朗日函数

$$
\mathcal{L}(A,b,x,\lambda)=\mathcal{L}(x)+\lambda^T(b-Ax).
$$

对 $x$ 求驻点条件：$\frac{\partial\mathcal{L}}{\partial x}-\lambda^T A=0$，即 $A^T\lambda=(\frac{\partial\mathcal{L}}{\partial x})^T$。因此伴随方程是 $A^T\lambda=v$，其中 $v$ 是上游梯度。对 $A$ 和 $b$ 的梯度由隐函数定理给出：

$$
\frac{\partial\mathcal{L}}{\partial b}=\lambda,\qquad
\frac{\partial\mathcal{L}}{\partial A}=-\lambda x^{*T}.
$$

如果让自动微分穿透求解器，内存消耗为 $O(HT)$；伴随方法把它降到 $O(H)$。这是伴随方法相对于离散-优化的核心优势。

**推导 2：中心化数据的SVD与PCA的对应。** 设 $X_c$ 为 $X$ 按列中心化后的矩阵。其SVD为 $X_c=U\Sigma V^T$。协方差矩阵为

$$
C=\frac{1}{n-1}X_c^TX_c
=\frac{1}{n-1}V\Sigma^T U^T U\Sigma V^T
=\frac{1}{n-1}V\Sigma^T\Sigma V^T.
$$

因此 $C$ 的特征向量为 $V$ 的列，特征值为 $\frac{\sigma_i^2}{n-1}$。主成分方向是 $V$ 的列，投影得分为 $U\Sigma$。

**推导 3：中心化与未中心化的平方和分解。** 设数据矩阵 $X$ 的均值为 $\bar{x}$（按列）。定义总平方和、均值贡献和波动贡献：

$$
s_{tot}^2=\sum_{i=1}^n\|x_i\|^2,\qquad
s_{mean}^2=n\|\bar{x}\|^2,\qquad
s_{err}^2=\sum_{i=1}^n\|x_i-\bar{x}\|^2.
$$

则有

$$
s_{tot}^2=s_{mean}^2+s_{err}^2.
$$

未中心化时PCA分析的是 $s_{tot}^2$，中心化后PCA分析的是 $s_{err}^2$。当 $s_{mean}^2\gg s_{err}^2$ 时（均值远离原点），PC1会主要捕获 $s_{mean}^2$，而不是数据真正的变异结构。

**推导 4：LoRA的参数量与初始化。** 原权重参数量为 $dk$。LoRA参数化为 $A\in\mathbb{R}^{r\times k}$，$B\in\mathbb{R}^{d\times r}$，参数量为 $r(d+k)$。当 $r\ll\min(d,k)$ 时，$r(d+k)\ll dk$。初始化 $B=0$ 使得 $BA=0$，训练从 $\Delta W=0$ 开始。若 $A$ 和 $B$ 都随机初始化，$\Delta W$ 会是一个随机低秩矩阵，破坏预训练模型的初始性能。

**推导 5：条件数在三种场景中的出场。** 在可微求解器中，条件数决定 $A^T\lambda=v$ 的求解误差，从而影响梯度质量。在PCA中，协方差矩阵的条件数决定特征值分解的数值稳定性（但实际用SVD更稳定）。在LoRA中，条件数不直接出现，但低秩假设的有效性依赖于权重更新矩阵的奇异值衰减。这样，4.3节的条件数概念在4.4节以三种不同的形式再次出场。

In [12]:
# 4.4 输入与独立参考
# TODO：填写本节输入、参数、种子和独立参考；不得用待验证算法自身输出冒充真值。
# 在这里补充代码；完成后删除这条占位提示。
print('4.4 输入与独立参考：未完成，请填写并实际运行。')


4.4 输入与独立参考：未完成，请填写并实际运行。


In [13]:
# 4.4 方法实现与运行
# TODO：用小矩阵核对求解器梯度与有限差分，并画PCA重构误差；LoRA只要求维度与参数量的小例子。
# 在这里补充代码；完成后删除这条占位提示。
print('4.4 方法实现与运行：未完成，请填写并实际运行。')


4.4 方法实现与运行：未完成，请填写并实际运行。


In [14]:
# 4.4 误差图表与比较
# TODO：记录梯度方向、差分步长、误差及数据是否中心化，给出低秩参数量计算。
# 在这里补充代码；完成后删除这条占位提示。
print('4.4 误差图表与比较：未完成，请填写并实际运行。')


4.4 误差图表与比较：未完成，请填写并实际运行。


### 结果解释与本节结论

应提供的证据：记录梯度方向、差分步长、误差及数据是否中心化，给出低秩参数量计算；比较两种LoRA初始化方案的损失曲线。


**本节结论：** 矩阵方法在AI中有三种典型嵌入方式：可微求解器嵌入训练循环，PCA嵌入数据预处理，LoRA嵌入参数高效微调。在三种场景中，4.1-4.3反复出现的条件数以不同形式再次出场：可微求解器中条件数决定梯度质量，PCA中条件数影响分解的数值稳定性，LoRA中低秩假设的有效性依赖于奇异值衰减。理解这些联系，才能正确使用矩阵方法并识别其边界。

### 易错点与讨论

待核验的风险：LoRA不等于对预训练权重直接做截断SVD；PCA未中心化会改变解释。

**补充讨论：**

1. **可微求解器的条件数风险**：当 $A$ 病态时，$A^T\lambda=v$ 同样病态，伴随求解的误差会被放大。条件数越大，梯度越不稳定。这是4.3节条件数概念在训练循环中的延续。
2. **PCA中心化的必要性**：未中心化时PC1会同时捕获均值贡献和波动贡献，当均值远离原点时会误读主成分方向。中心化后PCA才真正分析数据围绕均值的变异结构。
3. **LoRA初始化的多样性与动力学差异**：Init[A]和Init[B]的训练动力学不同；非零初始化也可行。初始化不是“随便选一个为零就行”，它影响训练效率。
4. **LoRA与截断SVD的区别**：标准LoRA的 $A$ 和 $B$ 是通过训练学到的，初始时 $\Delta W=0$；截断SVD是对已有权重的压缩或近似。二者是不同的操作。

**随堂提问：** 设计一道检查AI关于PCA或LoRA描述的题，给出维度和数学依据。设计一个病态矩阵，验证可微求解器梯度受条件数影响的现象。


**引用来源：**

[1] Gould, S. *Lecture notes on differentiable least squares*. Australian National University, 2022. 用于可微求解器伴随方法推导与PyTorch实现.[citation:18]

[2] Wang, Z., et al. *Sparse Linear Algebra in PyTorch*. arXiv:2402.01855, 2024. 用于线性求解反向传播的矩阵形式推导.[citation:10]

[3] IBM Corporation. *IBM SPSS Modeler Subscription: PCA 构建选项*. 2021. 用于PCA中心化必要性的说明.[citation:19]

[4] Hu, E., et al. *LoRA: Low-Rank Adaptation of Large Language Models*. arXiv:2106.09685, 2021. 用于LoRA核心公式、初始化策略与缩放因子.[citation:20]

[5] Meng, F., et al. *PiSSA: Principal Singular Values and Singular Vectors Adaptation of Large Language Models*. NeurIPS, 2024. 用于PiSSA用截断SVD初始化LoRA的方法[citation:5][citation:13].

### 4.4 实际运行记录

【待填写】运行日期、完整环境/新增依赖版本、代码状态、输出位置、误差与结论。对照上面的证据要求逐项解释；未运行写未运行。AI辅助内容及人工改动记录在本章模板-A组-AI对话记录.md。

## 回到开场：用证据回答

**本章核心问题：** Ax=b如何高效求解？矩阵的本质结构是什么？

【待填写】先回答漫画提出的具体问题，再连接到本章核心问题。保留原预测，指出支持结论的推导、Notebook小节和实际输出，写明比喻的局限。

## 本章小结与课后习题

围绕 **LU、CG、SVD** 各写一段：它解决什么问题、适用条件是什么、最容易混淆什么。其他方法说明与这三个概念的联系。

【待填写】本章小结。

课后题按四节对应填写到[模板-A组-习题.md](A2组-习题.md)，参考解答单独放在`模板-A组-习题参考解答.md`。

## 选读边界

GMRES、QR迭代和稀疏存储细节移出必修主线；后向误差仅在残差讨论中给出必要定义，完整分析选读。

选读不计入本章8学时和最低完成要求。若添加附录，只在此处填写已有材料的名称和链接，不为了占位增加空文档。

## 参考资料、图片来源与AI使用

| 支撑的主张/公式/图片 | 作者、标题、版本/年份 | 页码/章节/链接 | 本组人工核对 |
| --- | --- | --- | --- |
| LoRA的低秩参数增量（4.4） | Hu等，LoRA: Low-Rank Adaptation of Large Language Models，2021 | [原论文](https://arxiv.org/abs/2106.09685) | 阅读起点；待本组核对具体表述 |
| 待填写 | 待填写 | 待填写 | 未核对 |

漫画为仓库初始化助手绘制的概念示意，A组需结合最终正文核对并记录修改。AI建议与人工结论分开，本章对话分别记录到`模板-A组-AI对话记录.md`、`模板-B组-AI对话记录.md`、`模板-C组-AI对话记录.md`；A/B/C按轮换角色填写。
